# B2S 04 - AndinaLog IoT Telemetry

Conversion auditada de lecturas IoT desde Bronze hacia un Silver compacto, una cuarentena investigable
y un reporte de calidad con una fila por regla. Bronze se lee sin modificarse. WMS Orders, Productos y
Flota Silver se consultan unicamente para controles de correspondencia y coherencia de negocio.

**Entidad:** lectura de telemetria de cabina. **Granularidad:** una fila por `viaje_id` y `timestamp`
normalizados, con clave de lectura unica en Silver. Las desviaciones termicas operacionales no son
errores de calidad por si mismas.

**Alcance de esta entrega:** Bronze `datos/bronze/andinalog_iot_telemetry.csv`; Silver
`datos/silver/andinalog_iot_telemetry_silver.csv`; cuarentena
`datos/quarantine/andinalog_iot_telemetry_quarantine.csv`; reporte de calidad
`datos/quality/andinalog_iot_telemetry_reporte_calidad.csv`; informe
`informes/bronze_silver/Informe_B2S_04_IoT_Telemetry.md`.

**Exclusion expresa:** `datos/silver/andinalog_iot_telemetry_silver - Copy.csv` es una copia obsoleta
no entregable. No se lee, no se concilia, no se exporta y no participa de ningun control. Este notebook
usa exclusivamente la ruta canonica de Silver.

In [1]:
import hashlib
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 220)


def find_root():
    candidates = []
    if os.getenv("ANDINALOG_ROOT"):
        candidates.append(Path(os.environ["ANDINALOG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "datos" / "bronze").is_dir():
            return candidate
    raise FileNotFoundError("No se encontro el directorio datos/bronze")


def sha256_de_archivo(ruta):
    return hashlib.sha256(Path(ruta).read_bytes()).hexdigest()


def tabla_markdown(marco, indices=None):
    if indices is not None:
        marco = marco[indices]
    try:
        return marco.to_markdown(index=False)
    except Exception:
        return "\n".join(["```", marco.to_string(index=False), "```"])


ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
BRONZE_SHA256_ANTES = None

CONFIG = {
    "dataset": "andinalog_iot_telemetry",
    "etapa": "bronze_silver",
    "entidad": "lectura_iot_cabina",
    "granularidad": "una fila por viaje_id y timestamp normalizados",
    "clave_lectura": ["viaje_id", "timestamp"],
    "rutas": {
        "bronze": "datos/bronze/andinalog_iot_telemetry.csv",
        "wms_silver": "datos/silver/andinalog_wms_orders_silver.csv",
        "productos_silver": "datos/silver/andinalog_productos_silver.csv",
        "flota_silver": "datos/silver/andinalog_flota_silver.csv",
        "notebook": "notebooks/bronze_silver/04_iot_telemetry/B2S_04_AndinaLog_IoT_Telemetry.ipynb",
        "silver": "datos/silver/andinalog_iot_telemetry_silver.csv",
        "quarantine": "datos/quarantine/andinalog_iot_telemetry_quarantine.csv",
        "reporte_calidad": "datos/quality/andinalog_iot_telemetry_reporte_calidad.csv",
        "informe": "informes/bronze_silver/Informe_B2S_04_IoT_Telemetry.md",
        "copia_obsoleta_no_entregable": "datos/silver/andinalog_iot_telemetry_silver - Copy.csv",
    },
    "rutas_de_entrada": ["bronze", "wms_silver", "productos_silver", "flota_silver"],
    "rutas_de_salida": ["silver", "quarantine", "reporte_calidad", "informe"],
    "lectura": {"encoding": "utf-8", "dtype": "str", "keep_default_na": False},
    "columnas": {
        "obligatorias": [
            "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
            "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
            "desviacion_termica_flag", "desviacion_proximos_60min_flag",
        ],
        "identificadores": ["viaje_id", "order_id", "camion_id", "producto_id"],
        "numericas": ["temperatura_cabina_c", "humedad_cabina_pct"],
        "binarias": ["desviacion_termica_flag", "desviacion_proximos_60min_flag"],
        "fecha": "timestamp",
        "unidad_temperatura": "temp_unit",
        "permitir_extra": False,
    },
    "nombre_interno": {
        "temperatura_cabina_c": "temperatura_c",
        "humedad_cabina_pct": "humedad_pct",
        "desviacion_termica_flag": "flag_actual",
        "desviacion_proximos_60min_flag": "flag_futuro",
    },
    "formatos_identificador": {
        "viaje_id": r"^VIA-\d{5}$",
        "order_id": r"^ORD-2026-\d{5}$",
        "camion_id": r"^CAM-\d{2}$",
        "producto_id": r"^PROD-\d{3}$",
    },
    "formato_fecha": {
        "regex": r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$",
        "format": "%Y-%m-%d %H:%M:%S",
    },
    "zonas_horarias": {"sin_zona": "America/La_Paz", "silver": "UTC"},
    "unidades_temperatura": {"canonica": "C", "convertible": "F", "no_esperada": "K"},
    "conversion_fahrenheit_celsius": "(F - 32) * 5 / 9",
    "centinelas": {"temperatura_cabina_c": [-999], "humedad_cabina_pct": [-999]},
    "rangos_intrinsecos": {"humedad_cabina_pct": {"min": 0, "max": 100}},
    "rangos_termicos": {
        "aplicar_limite_tecnico": False,
        "motivo": "el plan propuso umbrales sin contrato; una desviacion operacional valida no es error de calidad",
    },
    "politica_duplicados": {
        "metodo": "cuarentena_de_todas_las_ocurrencias",
        "clave": ["t_viaje_id", "timestamp_original"],
        "motivo": "clave_lectura_duplicada_sin_criterio_de_desempate",
        "detalle": (
            "no se selecciona ganador porque no existe regla de negocio inequivoca; los duplicados "
            "exactos no se cuentan como un problema adicional independiente"
        ),
    },
    "politica_secuencia": {
        "metodo": "evaluar_sobre_serie_normalizada_ordenada_por_viaje_y_timestamp",
        "cadencia_esperada_min": 30,
        "orden_fisico_es_secuencia": False,
        "nota_orden_fisico": (
            "la posicion fisica en el CSV de Bronze se conserva solo en _fila_bronze y no se "
            "interpreta como secuencia cronologica"
        ),
    },
    "integridad_referencial": {
        "order_id": {"fuente": "wms_silver", "sin_correspondencia": "informativa"},
        "producto_id": {"fuente": "productos_silver", "sin_correspondencia": "informativa"},
        "camion_id": {"fuente": "flota_silver", "sin_correspondencia": "informativa"},
        "contradiccion_con_orden_wms": "error_referencial_bloqueante",
        "criterio": (
            "una lectura valida no pasa a cuarentena porque una fuente auxiliar no tenga clave "
            "utilizable; solo es bloqueante la contradiccion con una referencia existente"
        ),
    },
    "imputaciones": {
        "habilitadas": False,
        "metodos_prohibidos": ["media", "mediana", "moda", "ffill", "bfill", "informacion_futura"],
        "motivo": (
            "no existe regla reproducible, inequivoca y compatible con el dominio para imputar "
            "temperatura ni humedad ausentes; la ausencia se conserva y se reporta"
        ),
    },
    "motivos_transformacion": {
        "fecha_local_bolivia_convertida_utc": "timestamp local sin zona interpretado en America/La_Paz y convertido a UTC",
        "normalizacion_identificador:viaje_id": "correccion determinista de representacion con strip y upper en viaje_id",
        "normalizacion_identificador:order_id": "correccion determinista de representacion con strip y upper en order_id",
        "normalizacion_identificador:camion_id": "correccion determinista de representacion con strip y upper en camion_id",
        "normalizacion_identificador:producto_id": "correccion determinista de representacion con strip y upper en producto_id",
        "temperatura_fahrenheit_convertida_celsius": "temperatura Fahrenheit convertida a Celsius con la formula exacta",
    },
    "columnas_constantes_documentadas": {
        "silver": {
            "temp_unit": "Celsius es la unidad canonica; la columna declara la unidad de todas las filas",
            "calidad_estado": "en Silver toda fila es valida por construccion; el enrutamiento se aplica antes de proyectar",
        },
        "quarantine": {
            "calidad_estado": "en cuarentena toda fila esta en cuarentena por construccion",
            "motivos_imputacion": "imputacion deshabilitada en config; la columna deja constancia explicita de que no hubo imputaciones",
        },
    },
    "columnas_requeridas_por_gold": [
        "timestamp", "viaje_id", "order_id", "camion_id", "producto_id", "_fila_bronze",
        "temperatura_cabina_c", "humedad_cabina_pct", "desviacion_termica_flag",
        "desviacion_proximos_60min_flag",
    ],
    "columnas_requeridas_por_gold_opcionales": ["errores_bloqueantes", "fue_imputada"],
    "semilla": None,
    "nota_semilla": "el pipeline es deterministico y no usa aleatoriedad; la semilla no aplica",
}

print("Raiz:", ROOT)
print("Ejecucion UTC:", EXECUTED_AT_UTC)
print("Contradicciones documentadas:", 8)

Raiz: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Ejecucion UTC: 2026-09-26T07:00:05.461037+00:00
Contradicciones documentadas: 8


In [2]:
CONFIG["reglas"] = [
    {"regla_id": "IOT-CNV-001", "columna_evaluada": "temperatura_cabina_c;humedad_cabina_pct;desviacion_termica_flag;desviacion_proximos_60min_flag",
     "dimension_calidad": "exactitud", "severidad": "alta", "bloqueante": True, "informativa": False,
     "evaluable": "todas", "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Campo numerico no parseable o indicador fuera del dominio binario {0,1} en el valor recibido"},
    {"regla_id": "IOT-UNI-001", "columna_evaluada": "temp_unit", "dimension_calidad": "exactitud",
     "severidad": "informativa", "bloqueante": False, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "conversion_exacta_fahrenheit_a_celsius",
     "descripcion_regla": "Unidad de temperatura Fahrenheit convertida a Celsius con la formula exacta (F-32)*5/9"},
    {"regla_id": "IOT-UNI-002", "columna_evaluada": "temp_unit", "dimension_calidad": "validez",
     "severidad": "critica", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Unidad Kelvin, no esperada operacionalmente en telemetria de cabina"},
    {"regla_id": "IOT-UNI-003", "columna_evaluada": "temp_unit", "dimension_calidad": "validez",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Unidad de temperatura distinta de C, F o K, incluida la unidad vacia"},
    {"regla_id": "IOT-SEN-001", "columna_evaluada": "temperatura_cabina_c", "dimension_calidad": "exactitud",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Centinela -999 declarado en config: no es una medicion de temperatura"},
    {"regla_id": "IOT-SEN-002", "columna_evaluada": "humedad_cabina_pct", "dimension_calidad": "exactitud",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Centinela -999 declarado en config: no es una medicion de humedad"},
    {"regla_id": "IOT-NUL-001", "columna_evaluada": "temperatura_cabina_c", "dimension_calidad": "completitud",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Temperatura ausente en el valor recibido; no existe regla de imputacion inequivoca"},
    {"regla_id": "IOT-NUL-002", "columna_evaluada": "humedad_cabina_pct", "dimension_calidad": "completitud",
     "severidad": "informativa", "bloqueante": False, "informativa": True, "evaluable": "todas",
     "accion_si_ocurre": "sin_imputar_por_ausencia_de_regla_inequivoca",
     "descripcion_regla": "Humedad ausente en el valor recibido; se conserva nula y se reporta, sin imputar"},
    {"regla_id": "IOT-RNG-001", "columna_evaluada": "humedad_cabina_pct", "dimension_calidad": "validez",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Humedad fuera del rango intrinseco 0-100 grados de porcentaje"},
    {"regla_id": "IOT-FMT-001", "columna_evaluada": "viaje_id;order_id;camion_id;producto_id",
     "dimension_calidad": "validez", "severidad": "alta", "bloqueante": True, "informativa": False,
     "evaluable": "todas", "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Identificador que no cumple el patron declarado en config"},
    {"regla_id": "IOT-IDN-001", "columna_evaluada": "viaje_id;order_id;camion_id;producto_id",
     "dimension_calidad": "consistencia", "severidad": "informativa", "bloqueante": False,
     "informativa": False, "evaluable": "todas", "accion_si_ocurre": "normalizacion_determinista_strip_upper",
     "descripcion_regla": "Identificador normalizado con strip y upper; correccion determinista de representacion"},
    {"regla_id": "IOT-FEC-001", "columna_evaluada": "timestamp", "dimension_calidad": "validez",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Timestamp que no cumple el formato textual declarado en config"},
    {"regla_id": "IOT-FEC-002", "columna_evaluada": "timestamp", "dimension_calidad": "validez",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "Fecha imposible o instante inexistente en America/La_Paz; no se puede fijar el instante"},
    {"regla_id": "IOT-CLA-001", "columna_evaluada": "viaje_id;timestamp", "dimension_calidad": "unicidad",
     "severidad": "alta", "bloqueante": True, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "envio_a_cuarentena_de_todas_las_ocurrencias",
     "descripcion_regla": "Clave de lectura viaje_id mas timestamp duplicada, sin criterio de desempate en config"},
    {"regla_id": "IOT-SEQ-001", "columna_evaluada": "timestamp", "dimension_calidad": "consistencia",
     "severidad": "media", "bloqueante": False, "informativa": False, "evaluable": "timestamp_valido",
     "accion_si_ocurre": "sin_incidencias_tras_ordenar",
     "descripcion_regla": "Serie fuera de orden cronologico dentro del viaje, evaluada despues de normalizar y ordenar por viaje_id y timestamp"},
    {"regla_id": "IOT-SEQ-002", "columna_evaluada": "timestamp", "dimension_calidad": "consistencia",
     "severidad": "informativa", "bloqueante": False, "informativa": True, "evaluable": "timestamp_valido",
     "accion_si_ocurre": "informativa_hueco_de_cadencia",
     "descripcion_regla": "Intervalo entre lecturas consecutivas del mismo viaje mayor que la cadencia observada; hueco informativo, no bloqueante"},
    {"regla_id": "IOT-REF-001", "columna_evaluada": "order_id", "dimension_calidad": "integridad_referencial",
     "severidad": "informativa", "bloqueante": False, "informativa": True, "evaluable": "todas",
     "accion_si_ocurre": "informativa_sin_enriquecimiento",
     "descripcion_regla": "order_id sin correspondencia en WMS Silver: falta informativa de enriquecimiento, no error intrinseco del sensor"},
    {"regla_id": "IOT-REF-002", "columna_evaluada": "producto_id", "dimension_calidad": "integridad_referencial",
     "severidad": "informativa", "bloqueante": False, "informativa": True, "evaluable": "todas",
     "accion_si_ocurre": "informativa_sin_enriquecimiento",
     "descripcion_regla": "producto_id sin correspondencia en Productos Silver: falta informativa de enriquecimiento"},
    {"regla_id": "IOT-REF-003", "columna_evaluada": "camion_id", "dimension_calidad": "integridad_referencial",
     "severidad": "informativa", "bloqueante": False, "informativa": True, "evaluable": "todas",
     "accion_si_ocurre": "informativa_sin_enriquecimiento",
     "descripcion_regla": "camion_id sin correspondencia en Flota Silver: falta informativa de enriquecimiento"},
    {"regla_id": "IOT-REF-004", "columna_evaluada": "producto_id;camion_id",
     "dimension_calidad": "integridad_referencial", "severidad": "alta", "bloqueante": True,
     "informativa": False, "evaluable": "wms_coincidente", "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "producto_id o camion_id que contradice la orden WMS existente: error referencial bloqueante"},
    {"regla_id": "IOT-OPE-001", "columna_evaluada": "desviacion_termica_flag",
     "dimension_calidad": "consistencia", "severidad": "media", "bloqueante": True, "informativa": False,
     "evaluable": "desviacion_comparable", "accion_si_ocurre": "envio_a_cuarentena",
     "descripcion_regla": "desviacion_termica_flag incoherente con la temperatura medida frente a la temperatura de conservacion y tolerancia del producto; una desviacion operacional valida no es error"},
    {"regla_id": "IOT-IMP-001", "columna_evaluada": "todas", "dimension_calidad": "completitud",
     "severidad": "informativa", "bloqueante": False, "informativa": False, "evaluable": "todas",
     "accion_si_ocurre": "no_imputada_por_config", "no_aplicable": True,
     "descripcion_regla": "Ninguna variable fue imputada: no se uso media, mediana, moda, ffill, bfill ni informacion futura"},
]

CONFIG["columnas_silver"] = [
    "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
    "_fila_bronze", "timestamp_original", "temp_unit_original", "temperatura_cabina_c_original",
    "motivos_transformacion", "conteo_transformaciones", "banderas_informativas", "calidad_estado",
]

CONFIG["columnas_cuarentena"] = [
    "_fila_bronze", "errores_bloqueantes", "calidad_motivo", "calidad_estado",
    "clave_lectura_bronze", "clave_lectura_normalizada",
    "timestamp_original", "viaje_id_original", "order_id_original", "camion_id_original",
    "producto_id_original", "temperatura_cabina_c_original", "temp_unit_original",
    "humedad_cabina_pct_original", "desviacion_termica_flag_original",
    "desviacion_proximos_60min_flag_original",
    "timestamp_utc_normalizado", "temperatura_c_normalizada",
    "lectura_clave_duplicada", "ocurrencias_clave_lectura",
    "corresponde_wms_silver", "corresponde_producto_silver", "corresponde_flota_silver",
    "motivos_transformacion", "motivos_imputacion", "banderas_informativas",
    "conteo_transformaciones", "conteo_errores_bloqueantes", "conteo_banderas_informativas",
]

CONFIG["columnas_reporte_calidad"] = [
    "dataset", "etapa", "regla_id", "columna_evaluada", "dimension_calidad", "descripcion_regla",
    "tipo_resultado", "severidad", "filas_evaluadas", "filas_afectadas", "porcentaje_afectado",
    "accion_aplicada", "filas_silver", "filas_cuarentena", "estado_regla", "evidencia",
]

CONFIG["contradicciones_plan"] = [
    "hay 80 temperaturas vacias y 120 centinelas -999.0; el plan habia indicado ausencia de ambos",
    "hay 100 humedades vacias, no 951",
    "hay 15 fechas imposibles aunque todas cumplen el patron textual",
    "hay 230 filas duplicadas exactas y 240 filas en 120 claves de lectura duplicadas",
    "la frecuencia regular observada es 30 minutos, no 20-21",
    "no se aplican limites termicos -50/50 o -10/40 porque no existe contrato que los sustente",
    "las faltas referenciales se mantienen informativas, no bloqueantes, por no ser errores intrinsecos de sensor",
    "las inversiones de tiempo del orden fisico del CSV no son un error de la serie de lecturas",
]

PATHS = {nombre: ROOT / relativo for nombre, relativo in CONFIG["rutas"].items()}
PATHS["reporte_calidad"].parent.mkdir(parents=True, exist_ok=True)
PATHS["informe"].parent.mkdir(parents=True, exist_ok=True)

RUTAS_EXCLUIDAS = {CONFIG["rutas"]["copia_obsoleta_no_entregable"]}
for nombre in CONFIG["rutas_de_entrada"] + CONFIG["rutas_de_salida"]:
    assert CONFIG["rutas"][nombre] not in RUTAS_EXCLUIDAS, "la copia obsoleta no puede ser entrada ni salida"
    assert "copy" not in CONFIG["rutas"][nombre].lower(), "ninguna ruta activa puede apuntar a la copia obsoleta"

print("Reglas de calidad declaradas:", len(CONFIG["reglas"]))
print("Columnas Silver declaradas:", len(CONFIG["columnas_silver"]))
print("Columnas de cuarentena declaradas:", len(CONFIG["columnas_cuarentena"]))
print("Columnas del reporte de calidad:", len(CONFIG["columnas_reporte_calidad"]))
print("Rutas excluidas (copia obsoleta no entregable):", sorted(RUTAS_EXCLUIDAS))

Reglas de calidad declaradas: 22
Columnas Silver declaradas: 18
Columnas de cuarentena declaradas: 29
Columnas del reporte de calidad: 16
Rutas excluidas (copia obsoleta no entregable): ['datos/silver/andinalog_iot_telemetry_silver - Copy.csv']


In [3]:
BRONZE_SHA256_ANTES = sha256_de_archivo(PATHS["bronze"])
COPIA_OBSOLETA_EXISTE = PATHS["copia_obsoleta_no_entregable"].exists()
COPIA_OBSOLETA_SHA256_ANTES = (sha256_de_archivo(PATHS["copia_obsoleta_no_entregable"])
                               if COPIA_OBSOLETA_EXISTE else None)
COPIA_OBSOLETA_BYTES = (PATHS["copia_obsoleta_no_entregable"].stat().st_size
                        if COPIA_OBSOLETA_EXISTE else 0)

bronze = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])
wms_silver = pd.read_csv(PATHS["wms_silver"], **CONFIG["lectura"])
productos_silver = pd.read_csv(PATHS["productos_silver"], **CONFIG["lectura"])
flota_silver = pd.read_csv(PATHS["flota_silver"], **CONFIG["lectura"])

perfil_ts = pd.to_datetime(bronze["timestamp"], format=CONFIG["formato_fecha"]["format"], errors="coerce")
perfil_num = {columna: pd.to_numeric(bronze[columna], errors="coerce")
              for columna in CONFIG["columnas"]["numericas"] + CONFIG["columnas"]["binarias"]}
perfil_utc = perfil_ts.dt.tz_localize(
    CONFIG["zonas_horarias"]["sin_zona"], ambiguous="NaT", nonexistent="NaT"
).dt.tz_convert(CONFIG["zonas_horarias"]["silver"])
perfil_viaje = bronze["viaje_id"].str.strip().str.upper()
perfil_clave = pd.DataFrame({"viaje_id": perfil_viaje, "timestamp": bronze["timestamp"].str.strip()})

delta_fisico = perfil_ts.groupby(perfil_viaje, sort=False).diff().dt.total_seconds().div(60)
serie_ordenada = pd.DataFrame({"viaje_id": perfil_viaje, "ts": perfil_utc}).sort_values(
    ["viaje_id", "ts"], kind="stable")
delta_logico = serie_ordenada.groupby("viaje_id", sort=False)["ts"].diff().dt.total_seconds().div(60)

duplicadas = perfil_clave.duplicated(keep=False)
PERFIL = {
    "sha256_bronze_antes": BRONZE_SHA256_ANTES,
    "filas": len(bronze),
    "columnas": len(bronze.columns),
    "nombres_columnas": bronze.columns.tolist(),
    "tipos_recibidos": bronze.dtypes.astype(str).to_dict(),
    "vacios": bronze.eq("").sum().to_dict(),
    "duplicados_exactos_filas": int(bronze.duplicated(keep=False).sum()),
    "duplicados_lectura_filas": int(duplicadas.sum()),
    "duplicados_lectura_claves": int(perfil_clave.loc[duplicadas].drop_duplicates().shape[0]),
    "duplicados_no_exactos_filas": int((duplicadas & ~bronze.duplicated(keep=False)).sum()),
    "espacios_identificadores": {
        columna: int(bronze[columna].ne(bronze[columna].str.strip()).sum())
        for columna in CONFIG["columnas"]["identificadores"]
    },
    "unidades_observadas": bronze["temp_unit"].value_counts(dropna=False).to_dict(),
    "centinelas": {columna: int(perfil_num[columna].isin(valores).sum())
                   for columna, valores in CONFIG["centinelas"].items()},
    "fechas_formato_reconocido": int(bronze["timestamp"].str.match(CONFIG["formato_fecha"]["regex"]).sum()),
    "fechas_invalidas": int(perfil_utc.isna().sum()),
    "valores_fecha_invalida": sorted(bronze.loc[perfil_utc.isna(), "timestamp"].unique().tolist()),
    "frecuencia_minutos_ordenada": delta_logico.value_counts().sort_index().to_dict(),
    "inversiones_orden_fisico_archivo": int(delta_fisico.lt(0).sum()),
    "inversiones_tras_ordenar_por_viaje": int(delta_logico.lt(0).sum()),
    "lecturas_por_viaje": bronze.groupby(perfil_viaje).size().value_counts().sort_index().to_dict(),
    "rangos_numericos_raw": {columna: [float(serie.min()), float(serie.max())]
                             for columna, serie in perfil_num.items()},
    "humedad_fuera_de_rango_detallada": sorted(
        bronze.loc[perfil_num["humedad_cabina_pct"].notna()
                   & ~perfil_num["humedad_cabina_pct"].between(0, 100), "humedad_cabina_pct"].unique().tolist()),
}

print("Perfil Bronze obtenido del archivo real")
for clave, valor in PERFIL.items():
    print(f"{clave}: {valor}")
print()
print("Copia obsoleta", CONFIG["rutas"]["copia_obsoleta_no_entregable"],
      "| existe en disco:", PATHS["copia_obsoleta_no_entregable"].exists(),
      "| leida:", False, "| en conciliacion:", False, "| entregable:", False)

Perfil Bronze obtenido del archivo real
sha256_bronze_antes: edb7afe2f7fb836e59fe605d30c88b3b5b13a6d8ab2ec0b37f206a14e58de6bf
filas: 28920
columnas: 10
nombres_columnas: ['timestamp', 'viaje_id', 'order_id', 'camion_id', 'producto_id', 'temperatura_cabina_c', 'temp_unit', 'humedad_cabina_pct', 'desviacion_termica_flag', 'desviacion_proximos_60min_flag']
tipos_recibidos: {'timestamp': 'str', 'viaje_id': 'str', 'order_id': 'str', 'camion_id': 'str', 'producto_id': 'str', 'temperatura_cabina_c': 'str', 'temp_unit': 'str', 'humedad_cabina_pct': 'str', 'desviacion_termica_flag': 'str', 'desviacion_proximos_60min_flag': 'str'}
vacios: {'timestamp': 0, 'viaje_id': 0, 'order_id': 0, 'camion_id': 0, 'producto_id': 0, 'temperatura_cabina_c': 80, 'temp_unit': 0, 'humedad_cabina_pct': 100, 'desviacion_termica_flag': 0, 'desviacion_proximos_60min_flag': 0}
duplicados_exactos_filas: 230
duplicados_lectura_filas: 240
duplicados_lectura_claves: 120
duplicados_no_exactos_filas: 10
espacios_identificado

In [4]:
MARCAS = {}
MARCAS_EVALUABLE = {}
MARCAS_TRANSFORMACION = {}
INDICE_BASE = bronze.index


def como_bool(mascara):
    if isinstance(mascara, pd.Series):
        return mascara.fillna(False).to_numpy(dtype=bool)
    return np.asarray(mascara, dtype=bool)


def registro(regla_id, mascara):
    MARCAS[regla_id] = como_bool(mascara)


def evaluable(clave, mascara):
    MARCAS_EVALUABLE[clave] = como_bool(mascara)


def transformacion(nombre, mascara):
    MARCAS_TRANSFORMACION[nombre] = como_bool(mascara)


def componer(mascaras, tokens):
    if not tokens:
        return pd.Series("", index=INDICE_BASE, dtype=object)
    partes = pd.DataFrame({
        token: pd.Series(como_bool(mascara), index=INDICE_BASE).map({True: token, False: ""})
        for mascara, token in zip(mascaras, tokens)
    })
    return partes.apply(lambda fila: " | ".join([valor for valor in fila if valor]), axis=1)


def normaliza(serie):
    return serie.str.strip().str.upper()


def texto_si_no(serie):
    return serie.map({True: "si", False: "no"})


def formatea_utc(serie):
    return serie.dt.strftime("%Y-%m-%d %H:%M:%S%z").str.replace(
        r"([+-])(\d{2})(\d{2})$", r"\1\2:\3", regex=True)


def validar_contrato_entrada(df):
    df = df.copy()
    esperadas = set(CONFIG["columnas"]["obligatorias"])
    recibidas = set(df.columns)
    faltantes = sorted(esperadas - recibidas)
    extra = sorted(recibidas - esperadas)
    if faltantes or (extra and not CONFIG["columnas"]["permitir_extra"]):
        raise ValueError(f"Contrato invalido; faltantes={faltantes}, extra={extra}")
    return df


def estructurar(df):
    df = df.copy().reset_index(drop=True)
    df["_fila_bronze"] = range(2, len(df) + 2)
    for columna in CONFIG["columnas"]["obligatorias"]:
        df[f"{columna}_original"] = df[columna]
    for columna in ["errores_bloqueantes", "banderas_informativas",
                    "motivos_transformacion", "motivos_imputacion", "calidad_motivo"]:
        df[columna] = ""
    return df


def normalizar_identificadores_y_unidad(df):
    df = df.copy()
    mascara_idn = pd.Series(False, index=df.index)
    for columna in CONFIG["columnas"]["identificadores"]:
        df[f"t_{columna}"] = normaliza(df[f"{columna}_original"])
        cambio = df[f"{columna}_original"].ne(df[f"t_{columna}"])
        transformacion(f"normalizacion_identificador:{columna}", cambio)
        mascara_idn |= cambio
    df["t_temp_unit"] = normaliza(df["temp_unit_original"])
    registro("IOT-IDN-001", mascara_idn)
    return df


def convertir_numericos_y_temperatura(df):
    df = df.copy()
    mascara_cnv = pd.Series(False, index=df.index)
    centinelas = {}
    for columna in CONFIG["columnas"]["numericas"] + CONFIG["columnas"]["binarias"]:
        interno = f"t_{CONFIG['nombre_interno'][columna]}"
        recibido = df[f"{columna}_original"]
        bruto = pd.to_numeric(recibido, errors="coerce")
        vacio = recibido.str.strip().eq("")
        mascara_cnv |= bruto.isna() & ~vacio
        if columna in CONFIG["columnas"]["binarias"]:
            mascara_cnv |= bruto.notna() & ~bruto.isin([0, 1])
        centinelas[columna] = bruto.isin(CONFIG["centinelas"].get(columna, []))
        df[interno] = bruto.mask(centinelas[columna])
    registro("IOT-CNV-001", mascara_cnv)
    registro("IOT-SEN-001", centinelas["temperatura_cabina_c"])
    registro("IOT-SEN-002", centinelas["humedad_cabina_pct"])

    canonica = CONFIG["unidades_temperatura"]["canonica"]
    convertible = CONFIG["unidades_temperatura"]["convertible"]
    no_esperada = CONFIG["unidades_temperatura"]["no_esperada"]
    unidad = df["t_temp_unit"]
    es_f = unidad.eq(convertible)
    es_k = unidad.eq(no_esperada)
    conocida = unidad.isin([canonica, convertible, no_esperada])
    registro("IOT-UNI-001", es_f)
    registro("IOT-UNI-002", es_k)
    registro("IOT-UNI-003", ~conocida)
    transformacion("temperatura_fahrenheit_convertida_celsius", es_f)

    temperatura = df["t_temperatura_c"].copy()
    temperatura.loc[es_f] = (temperatura.loc[es_f] - 32) * 5 / 9
    temperatura.loc[es_k | ~conocida] = np.nan
    df["t_temperatura_c"] = temperatura
    df["t_temp_unit"] = unidad.where(conocida)

    registro("IOT-NUL-001", df["temperatura_cabina_c_original"].str.strip().eq(""))
    registro("IOT-NUL-002", df["humedad_cabina_pct_original"].str.strip().eq(""))

    limites = CONFIG["rangos_intrinsecos"]["humedad_cabina_pct"]
    humedad = df["t_humedad_pct"]
    registro("IOT-RNG-001", humedad.notna() & ~humedad.between(limites["min"], limites["max"]))
    return df


def convertir_timestamp(df):
    df = df.copy()
    regla = CONFIG["formato_fecha"]
    registro("IOT-FEC-001", ~df["timestamp_original"].str.match(regla["regex"]))
    analizado = pd.to_datetime(df["timestamp_original"], format=regla["format"], errors="coerce")
    localizado = analizado.dt.tz_localize(
        CONFIG["zonas_horarias"]["sin_zona"], ambiguous="NaT", nonexistent="NaT")
    df["t_timestamp"] = localizado.dt.tz_convert(CONFIG["zonas_horarias"]["silver"])
    registro("IOT-FEC-002", df["t_timestamp"].isna())
    evaluable("timestamp_valido", df["t_timestamp"].notna())
    transformacion("fecha_local_bolivia_convertida_utc", df["t_timestamp"].notna())
    return df


def validar_clave_secuencia_y_flags(df):
    df = df.copy()
    mascara_fmt = pd.Series(False, index=df.index)
    for columna, patron in CONFIG["formatos_identificador"].items():
        mascara_fmt |= ~df[f"t_{columna}"].fillna("").str.match(patron)
    registro("IOT-FMT-001", mascara_fmt)

    clave = CONFIG["politica_duplicados"]["clave"]
    df["_clave_bronze"] = df["t_viaje_id"] + " | " + df["timestamp_original"].str.strip()
    df["_lectura_clave_duplicada"] = df.duplicated(clave, keep=False)
    df["_ocurrencias"] = df.groupby(clave)["_fila_bronze"].transform("size")
    registro("IOT-CLA-001", df["_lectura_clave_duplicada"])

    orden = df.sort_values(["t_viaje_id", "t_timestamp"], kind="stable")
    delta = orden.groupby("t_viaje_id", sort=False)["t_timestamp"].diff().dt.total_seconds().div(60)
    cadencia = CONFIG["politica_secuencia"]["cadencia_esperada_min"]
    fuera_de_orden = pd.Series(False, index=df.index)
    hueco = pd.Series(False, index=df.index)
    fuera_de_orden.loc[orden.index] = (delta < 0).to_numpy()
    hueco.loc[orden.index] = (delta > cadencia).to_numpy()
    registro("IOT-SEQ-001", fuera_de_orden)
    registro("IOT-SEQ-002", hueco)
    return df


def validar_integridad_referencial_y_operacional(df):
    df = df.copy()
    wms = wms_silver.assign(_orden=normaliza(wms_silver["order_id"])).set_index("_orden")
    productos = set(normaliza(productos_silver["producto_id"]))
    flota = set(normaliza(flota_silver["camion_id"]))

    corresponde_wms = df["t_order_id"].isin(wms.index)
    corresponde_producto = df["t_producto_id"].isin(productos)
    corresponde_camion = df["t_camion_id"].isin(flota)
    registro("IOT-REF-001", ~corresponde_wms)
    registro("IOT-REF-002", ~corresponde_producto)
    registro("IOT-REF-003", ~corresponde_camion)
    evaluable("wms_coincidente", corresponde_wms)

    pedido_producto = df["t_order_id"].map(normaliza(wms["producto_id"]))
    pedido_camion = df["t_order_id"].map(normaliza(wms["camion_id"]))
    registro("IOT-REF-004", corresponde_wms & (
        ~df["t_producto_id"].eq(pedido_producto) | ~df["t_camion_id"].eq(pedido_camion)))

    catalogo = productos_silver.assign(_producto=normaliza(productos_silver["producto_id"])).set_index("_producto")
    requerido = df["t_producto_id"].map(pd.to_numeric(catalogo["temperatura_conservacion_requerida_c"], errors="coerce"))
    tolerancia = df["t_producto_id"].map(pd.to_numeric(catalogo["tolerancia_temperatura_c"], errors="coerce"))
    temperatura = pd.to_numeric(df["t_temperatura_c"], errors="coerce")
    comparable = temperatura.notna() & requerido.notna() & tolerancia.notna()
    calculada = (temperatura < requerido - tolerancia) | (temperatura > requerido + tolerancia)
    evaluable("desviacion_comparable", comparable)
    registro("IOT-OPE-001", comparable & ~df["t_flag_actual"].astype("Float64").eq(calculada.astype("Float64")))
    df["_corresponde_wms"] = corresponde_wms
    df["_corresponde_producto"] = corresponde_producto
    df["_corresponde_camion"] = corresponde_camion
    return df


def imputar(df):
    df = df.copy()
    df["motivos_imputacion"] = ""
    if CONFIG["imputaciones"]["habilitadas"]:
        raise NotImplementedError("no hay regla de imputacion declarada en config")
    registro("IOT-IMP-001", pd.Series(False, index=df.index))
    return df


def asignar_calidad(df):
    df = df.copy()
    bloqueantes = [regla for regla in CONFIG["reglas"] if regla["bloqueante"]]
    informativas = [regla for regla in CONFIG["reglas"] if regla["informativa"]]
    df["errores_bloqueantes"] = componer(
        [MARCAS[regla["regla_id"]] for regla in bloqueantes], [regla["regla_id"] for regla in bloqueantes])
    df["banderas_informativas"] = componer(
        [MARCAS[regla["regla_id"]] for regla in informativas], [regla["regla_id"] for regla in informativas])
    nombres = list(CONFIG["motivos_transformacion"])
    df["motivos_transformacion"] = componer([MARCAS_TRANSFORMACION[nombre] for nombre in nombres], nombres)
    for origen, destino in [("motivos_transformacion", "conteo_transformaciones"),
                            ("errores_bloqueantes", "conteo_errores_bloqueantes"),
                            ("banderas_informativas", "conteo_banderas_informativas")]:
        df[destino] = df[origen].map(lambda valor: len([p for p in valor.split("|") if p.strip()]))
    descripciones = {regla["regla_id"]: regla["descripcion_regla"] for regla in CONFIG["reglas"]}
    df["calidad_motivo"] = df["errores_bloqueantes"].map(
        lambda valor: descripciones[valor.split(" | ")[0]] if valor else "")
    df["calidad_estado"] = "valida"
    df.loc[df["errores_bloqueantes"].ne(""), "calidad_estado"] = "cuarentena"
    return df


work = (
    bronze.pipe(validar_contrato_entrada)
    .pipe(estructurar)
    .pipe(normalizar_identificadores_y_unidad)
    .pipe(convertir_numericos_y_temperatura)
    .pipe(convertir_timestamp)
    .pipe(validar_clave_secuencia_y_flags)
    .pipe(validar_integridad_referencial_y_operacional)
    .pipe(imputar)
    .pipe(asignar_calidad)
)

silver = work.loc[work["calidad_estado"].eq("valida")].copy()
quarantine = work.loc[work["calidad_estado"].eq("cuarentena")].copy()

print("Enrutamiento por estado de calidad, calculado despues del tratamiento y la imputacion")
print(work["calidad_estado"].value_counts().to_string())
print()
print("Activaciones por regla sobre el dataframe de trabajo (las activaciones se solapan)")
for regla in CONFIG["reglas"]:
    total = int(MARCAS[regla["regla_id"]].sum())
    if total:
        print(f"  {regla['regla_id']}: {total}")
print()
print("Silver:", len(silver), "| cuarentena:", len(quarantine), "| Bronze:", len(bronze))

Enrutamiento por estado de calidad, calculado despues del tratamiento y la imputacion
calidad_estado
valida        28448
cuarentena      472

Activaciones por regla sobre el dataframe de trabajo (las activaciones se solapan)
  IOT-UNI-001: 50
  IOT-UNI-002: 5
  IOT-SEN-001: 120
  IOT-NUL-001: 80
  IOT-NUL-002: 100
  IOT-RNG-001: 15
  IOT-IDN-001: 50
  IOT-FEC-002: 15
  IOT-CLA-001: 240
  IOT-SEQ-002: 13
  IOT-REF-001: 4942
  IOT-REF-002: 3159
  IOT-REF-003: 1637

Silver: 28448 | cuarentena: 472 | Bronze: 28920


In [5]:
def proyectar_silver(df):
    salida = pd.DataFrame(index=df.index)
    salida["timestamp"] = df["t_timestamp"]
    for columna in CONFIG["columnas"]["identificadores"]:
        salida[columna] = df[f"t_{columna}"]
    salida["temperatura_cabina_c"] = pd.to_numeric(df["t_temperatura_c"], errors="coerce")
    salida["temp_unit"] = CONFIG["unidades_temperatura"]["canonica"]
    salida["humedad_cabina_pct"] = pd.to_numeric(df["t_humedad_pct"], errors="coerce")
    for columna in CONFIG["columnas"]["binarias"]:
        salida[columna] = pd.to_numeric(df[f"t_{CONFIG['nombre_interno'][columna]}"], errors="coerce").astype("Int64")
    salida["_fila_bronze"] = df["_fila_bronze"]
    salida["timestamp_original"] = df["timestamp_original"]
    salida["temp_unit_original"] = df["temp_unit_original"]
    salida["temperatura_cabina_c_original"] = df["temperatura_cabina_c_original"]
    salida["motivos_transformacion"] = df["motivos_transformacion"]
    salida["conteo_transformaciones"] = df["conteo_transformaciones"]
    salida["banderas_informativas"] = df["banderas_informativas"]
    salida["calidad_estado"] = df["calidad_estado"]
    return salida[CONFIG["columnas_silver"]]


def proyectar_cuarentena(df):
    salida = pd.DataFrame(index=df.index)
    salida["_fila_bronze"] = df["_fila_bronze"]
    salida["errores_bloqueantes"] = df["errores_bloqueantes"]
    salida["calidad_motivo"] = df["calidad_motivo"]
    salida["calidad_estado"] = df["calidad_estado"]
    salida["clave_lectura_bronze"] = df["_clave_bronze"]
    salida["timestamp_utc_normalizado"] = formatea_utc(df["t_timestamp"])
    salida["clave_lectura_normalizada"] = df["t_viaje_id"] + " | " + salida["timestamp_utc_normalizado"]
    for columna in CONFIG["columnas"]["obligatorias"]:
        salida[f"{columna}_original"] = df[f"{columna}_original"]
    salida["temperatura_c_normalizada"] = pd.to_numeric(df["t_temperatura_c"], errors="coerce")
    salida["lectura_clave_duplicada"] = df["_lectura_clave_duplicada"]
    salida["ocurrencias_clave_lectura"] = df["_ocurrencias"]
    salida["corresponde_wms_silver"] = texto_si_no(df["_corresponde_wms"])
    salida["corresponde_producto_silver"] = texto_si_no(df["_corresponde_producto"])
    salida["corresponde_flota_silver"] = texto_si_no(df["_corresponde_camion"])
    salida["motivos_transformacion"] = df["motivos_transformacion"]
    salida["motivos_imputacion"] = df["motivos_imputacion"]
    salida["banderas_informativas"] = df["banderas_informativas"]
    salida["conteo_transformaciones"] = df["conteo_transformaciones"]
    salida["conteo_errores_bloqueantes"] = df["conteo_errores_bloqueantes"]
    salida["conteo_banderas_informativas"] = df["conteo_banderas_informativas"]
    return salida[CONFIG["columnas_cuarentena"]]


def comprobar_ausencia_de_bloqueantes_en_silver(indices_silver):
    activos = {}
    for regla in CONFIG["reglas"]:
        if not regla["bloqueante"]:
            continue
        mascara = MARCAS[regla["regla_id"]]
        presentes = int(mascara[indices_silver.to_numpy()].sum())
        if presentes:
            activos[regla["regla_id"]] = presentes
    assert not activos, f"reglas bloqueantes activas en filas destinadas a Silver: {activos}"
    return True


silver_export = proyectar_silver(silver)
quarantine_export = proyectar_cuarentena(quarantine)

assert list(silver_export.columns) == CONFIG["columnas_silver"], "orden de columnas Silver distinto al declarado"
assert list(quarantine_export.columns) == CONFIG["columnas_cuarentena"], "orden de cuarentena distinto al declarado"
assert len(set(CONFIG["columnas_silver"])) == len(CONFIG["columnas_silver"]), "columnas Silver repetidas"
assert len(set(CONFIG["columnas_cuarentena"])) == len(CONFIG["columnas_cuarentena"]), "columnas de cuarentena repetidas"
control_bloqueantes = comprobar_ausencia_de_bloqueantes_en_silver(silver.index)

print("Ninguna regla bloqueante activa en las filas destinadas a Silver:", control_bloqueantes)
print("Silver: filas", len(silver_export), "columnas", len(silver_export.columns))
print("Cuarentena: filas", len(quarantine_export), "columnas", len(quarantine_export.columns))
print()
print("Silver")
print(silver_export.head(6).to_string(index=False))
print()
print("Cuarentena")
print(quarantine_export.head(6).to_string(index=False))

Ninguna regla bloqueante activa en las filas destinadas a Silver: True
Silver: filas 28448 columnas 18
Cuarentena: filas 472 columnas 29

Silver
                timestamp  viaje_id       order_id camion_id producto_id  temperatura_cabina_c temp_unit  humedad_cabina_pct  desviacion_termica_flag  desviacion_proximos_60min_flag  _fila_bronze  timestamp_original temp_unit_original temperatura_cabina_c_original             motivos_transformacion  conteo_transformaciones banderas_informativas calidad_estado
2026-08-11 18:11:00+00:00 VIA-00001 ORD-2026-05710    CAM-20    PROD-045                  2.19         C                72.6                        0                               0             2 2026-08-11 14:11:00                  C                          2.19 fecha_local_bolivia_convertida_utc                        1                               valida
2026-08-11 18:41:00+00:00 VIA-00001 ORD-2026-05710    CAM-20    PROD-045                  4.91         C                75.0       

In [6]:
SILVER_ETIQUETAS = set(silver.index)
CUARENTENA_ETIQUETAS = set(quarantine.index)
EXTRAS_EVIDENCIA = {}


def estado_regla_calculado(regla, afectadas):
    if regla.get("no_aplicable"):
        return "no_aplicable"
    if afectadas == 0:
        return "evaluada_sin_incidencias"
    if regla["bloqueante"]:
        return "envio_a_cuarentena"
    if regla["informativa"]:
        return "informativa_sin_cuarentena"
    return "aplicada_transformacion"


def construir_reporte_calidad():
    filas = []
    for regla in CONFIG["reglas"]:
        mascara = pd.Series(MARCAS[regla["regla_id"]], index=work.index)
        if regla.get("no_aplicable"):
            mascara_evaluable = pd.Series(False, index=work.index)
        elif regla["evaluable"] == "todas":
            mascara_evaluable = pd.Series(True, index=work.index)
        else:
            mascara_evaluable = pd.Series(MARCAS_EVALUABLE[regla["evaluable"]], index=work.index)
        efectiva = mascara & mascara_evaluable
        evaluadas = int(mascara_evaluable.sum())
        afectadas = int(efectiva.sum())
        en_silver = int((efectiva & work.index.isin(SILVER_ETIQUETAS)).sum())
        en_cuarentena = int((efectiva & work.index.isin(CUARENTENA_ETIQUETAS)).sum())
        extras = EXTRAS_EVIDENCIA.get(regla["regla_id"], "")
        if regla.get("no_aplicable"):
            accion = regla["accion_si_ocurre"]
        else:
            accion = regla["accion_si_ocurre"] if afectadas > 0 else "sin_incidencias"
        filas.append({
            "dataset": CONFIG["dataset"],
            "etapa": CONFIG["etapa"],
            "regla_id": regla["regla_id"],
            "columna_evaluada": regla["columna_evaluada"],
            "dimension_calidad": regla["dimension_calidad"],
            "descripcion_regla": regla["descripcion_regla"],
            "tipo_resultado": "conteo",
            "severidad": regla["severidad"],
            "filas_evaluadas": evaluadas,
            "filas_afectadas": afectadas,
            "porcentaje_afectado": round(afectadas / evaluadas, 6) if evaluadas else 0.0,
            "accion_aplicada": accion,
            "filas_silver": en_silver,
            "filas_cuarentena": en_cuarentena,
            "estado_regla": estado_regla_calculado(regla, afectadas),
            "evidencia": (f"denominador {evaluadas} filas"
                          + (" (no aplicable: no hay ninguna fila que evaluar)" if regla.get("no_aplicable") else "")
                          + f"; afectadas {afectadas}; "
                          f"porcentaje {round(afectadas / evaluadas, 6) if evaluadas else 0.0}; "
                          f"en Silver {en_silver}; en cuarentena {en_cuarentena}. {extras}"),
        })
    return pd.DataFrame(filas)[CONFIG["columnas_reporte_calidad"]]


claves_duplicadas = work.loc[work["_lectura_clave_duplicada"], "_clave_bronze"].nunique()
filas_duplicadas = int(work["_lectura_clave_duplicada"].sum())
duplicados_con_clave_duplicada = work.loc[work["_lectura_clave_duplicada"]]
marcas_duplicado_exacto = bronze.duplicated(keep=False).to_numpy()
exacto_en_clave_duplicada = int(
    marcas_duplicado_exacto[duplicados_con_clave_duplicada.index.to_numpy()].sum())
subconjunto_duplicado = bronze.assign(
    _v=bronze["viaje_id"].str.strip().str.upper(), _t=bronze["timestamp"].str.strip()
).loc[marcas_duplicado_exacto | work["_lectura_clave_duplicada"].to_numpy()]
grupos_duplicados = subconjunto_duplicado.groupby(["_v", "_t"], sort=False)
claves_con_diferencias = 0
filas_en_claves_con_diferencias = 0
for _, grupo in grupos_duplicados:
    distintas = grupo.drop_duplicates().shape[0]
    if distintas > 1:
        claves_con_diferencias += 1
        filas_en_claves_con_diferencias += len(grupo)
unidades = bronze["temp_unit"].str.strip().str.upper().value_counts().to_dict()
rango_por_unidad = {}
for unidad in unidades:
    valores = pd.to_numeric(bronze.loc[bronze["temp_unit"].str.strip().str.upper() == unidad,
                                        "temperatura_cabina_c"], errors="coerce")
    rango_por_unidad[unidad] = [float(valores.min()), float(valores.max())]
wms_coincidentes = int(MARCAS_EVALUABLE["wms_coincidente"].sum())
comparables = int(MARCAS_EVALUABLE["desviacion_comparable"].sum())
timestamps_validos = int(MARCAS_EVALUABLE["timestamp_valido"].sum())
desviaciones_operacionales = int(pd.to_numeric(bronze["desviacion_termica_flag"], errors="coerce").eq(1).sum())
referencias = {columna: bronze[columna].str.strip().str.upper()
                for columna in ["order_id", "producto_id", "camion_id"]}
sin_wms_unicos = int(bronze.loc[pd.Series(MARCAS["IOT-REF-001"], index=work.index), "order_id"]
                     .str.strip().str.upper().nunique())
sin_producto_unicos = int(bronze.loc[pd.Series(MARCAS["IOT-REF-002"], index=work.index), "producto_id"]
                          .str.strip().str.upper().nunique())
sin_camion_unicos = int(bronze.loc[pd.Series(MARCAS["IOT-REF-003"], index=work.index), "camion_id"]
                        .str.strip().str.upper().nunique())
imputaciones = int(MARCAS["IOT-IMP-001"].sum())

EXTRAS_EVIDENCIA.update({
    "IOT-UNI-001": (f"unidades observadas {unidades}; Fahrenheit convertible de forma exacta; "
                    f"rango Fahrenheit recibido {rango_por_unidad.get('F')}; las lecturas Fahrenheit que "
                    f"tambien fallan otra regla aparecen en cuarentena."),
    "IOT-UNI-002": f"unidades observadas {unidades}; rango Kelvin recibido {rango_por_unidad.get('K')}; Kelvin no es esperado operacionalmente.",
    "IOT-UNI-003": f"unidades observadas {unidades}; no hay unidades fuera de C, F o K.",
    "IOT-SEN-001": f"centinelas declarados en config {CONFIG['centinelas']['temperatura_cabina_c']}; no son mediciones.",
    "IOT-SEN-002": f"centinelas declarados en config {CONFIG['centinelas']['humedad_cabina_pct']}; ninguno observado en esta fuente.",
    "IOT-NUL-001": "temperaturas vacias en el valor recibido; no se imputo ninguna por falta de regla inequivoca.",
    "IOT-NUL-002": ("humedades vacias en el valor recibido; se conservan nulas en Silver con bandera informativa; "
                    "no se uso media, mediana, moda, ffill ni bfill."),
    "IOT-RNG-001": f"valores fuera de rango {PERFIL['humedad_fuera_de_rango_detallada']}; ninguno supera 100.",
    "IOT-FMT-001": "los cuatro identificadores cumplen los patrones declarados en config en todas las filas.",
    "IOT-IDN-001": f"espacios en el valor recibido por identificador {PERFIL['espacios_identificadores']}; normalizacion determinista y reversible.",
    "IOT-FEC-001": f"formato declarado {CONFIG['formato_fecha']['format']}; {PERFIL['fechas_formato_reconocido']} filas lo cumplen.",
    "IOT-FEC-002": f"valores imposibles {PERFIL['valores_fecha_invalida']}; el instante no es determinable sin una regla de correccion.",
    "IOT-CLA-001": (f"{claves_duplicadas} claves duplicadas en {filas_duplicadas} filas; "
                    f"{exacto_en_clave_duplicada} de esas filas son duplicado exacto de su clave y no se cuentan "
                    f"como un problema adicional; {claves_con_diferencias} claves tienen ocurrencias con valores "
                    f"diferentes entre si, en {filas_en_claves_con_diferencias} filas; "
                    f"motivo {CONFIG['politica_duplicados']['motivo']}."),
    "IOT-SEQ-001": (f"evaluada sobre la serie normalizada y ordenada por viaje_id y timestamp; "
                    f"{PERFIL['inversiones_tras_ordenar_por_viaje']} inversiones cronologicas; las "
                    f"{PERFIL['inversiones_orden_fisico_archivo']} inversiones del orden fisico del CSV de Bronze se "
                    f"documentan como caracteristica del archivo de entrada y no como error de las lecturas; "
                    f"_fila_bronze conserva la posicion fisica."),
    "IOT-SEQ-002": (f"cadencia observada {CONFIG['politica_secuencia']['cadencia_esperada_min']} minutos; "
                    f"intervalos de la serie ordenada {PERFIL['frecuencia_minutos_ordenada']}; los intervalos de "
                    f"0 minutos corresponden a claves duplicadas y los mayores son huecos informativos."),
    "IOT-REF-001": f"{sin_wms_unicos} order_id distintos sin correspondencia en WMS Silver; falta informativa, no error intrinseco.",
    "IOT-REF-002": f"{sin_producto_unicos} producto_id distintos sin correspondencia en Productos Silver; falta informativa de enriquecimiento.",
    "IOT-REF-003": f"{sin_camion_unicos} camion_id distintos sin correspondencia en Flota Silver; falta informativa de enriquecimiento.",
    "IOT-REF-004": f"denominador lecturas con orden WMS existente {wms_coincidentes}; contradicciones detectadas {int(MARCAS['IOT-REF-004'].sum())}.",
    "IOT-OPE-001": (f"denominador lecturas comparables {comparables}; desviaciones termicas operacionales observadas "
                    f"{desviaciones_operacionales}; una desviacion operacional valida no es error de calidad."),
    "IOT-IMP-001": (f"imputaciones deshabilitadas en config, por lo que la regla no es aplicable y no tiene "
                    f"denominador de evaluacion; metodos prohibidos {CONFIG['imputaciones']['metodos_prohibidos']}; "
                    f"filas imputadas {imputaciones}."),
})

reporte_calidad = construir_reporte_calidad()

assert len(reporte_calidad) == len(CONFIG["reglas"]), "el reporte debe tener una fila por regla"
assert reporte_calidad["regla_id"].is_unique, "regla_id debe ser unico y estable"
assert list(reporte_calidad.columns) == CONFIG["columnas_reporte_calidad"], "orden del reporte distinto al declarado"
for columna in ["filas_evaluadas", "filas_afectadas", "filas_silver", "filas_cuarentena"]:
    assert reporte_calidad[columna].dtype.kind in "iu", f"{columna} debe ser un entero"
assert (reporte_calidad["filas_silver"] + reporte_calidad["filas_cuarentena"]).equals(
    reporte_calidad["filas_afectadas"]), "cada fila afectada debe terminar en Silver o en cuarentena"
assert int(reporte_calidad.loc[reporte_calidad["estado_regla"].eq("envio_a_cuarentena"), "filas_silver"].sum()) == 0, \
    "ninguna regla bloqueante puede tener filas afectadas en Silver"
no_aplicables = reporte_calidad.loc[reporte_calidad["estado_regla"].eq("no_aplicable")]
assert len(no_aplicables) > 0, "debe existir al menos una regla no aplicable declarada en config"
assert no_aplicables["filas_evaluadas"].eq(0).all(), "una regla no aplicable no puede declarar filas evaluadas"
assert no_aplicables["accion_aplicada"].ne("sin_incidencias").all(), \
    "una regla no aplicable debe declarar su accion declarada en config, no una ausencia de incidencias"
sin_incidencias_evaluadas = reporte_calidad.loc[reporte_calidad["estado_regla"].eq("evaluada_sin_incidencias")]
assert sin_incidencias_evaluadas["filas_evaluadas"].gt(0).all(), \
    "toda regla evaluada sin incidencias debe tener un denominador de evaluacion mayor que cero"
tokens_informativos = sorted({token.strip() for valor in quarantine_export["banderas_informativas"]
                              for token in valor.split("|") if token.strip()})
tokens_informativos_silver = sorted({token.strip() for valor in silver_export["banderas_informativas"]
                                     for token in valor.split("|") if token.strip()})
informativos_declarados = sorted(regla["regla_id"] for regla in CONFIG["reglas"] if regla["informativa"])
assert set(tokens_informativos) <= set(informativos_declarados), \
    f"los tokens informativos de la cuarentena deben ser reglas informativas declaradas: {tokens_informativos}"
assert sorted(set(tokens_informativos) | set(tokens_informativos_silver)) == informativos_declarados, \
    f"la union de tokens informativos de ambas salidas debe ser exactamente las reglas informativas declaradas: {informativos_declarados}"

print("Reporte de calidad: una fila por regla")
print(reporte_calidad[["regla_id", "columna_evaluada", "dimension_calidad", "severidad", "filas_evaluadas",
                       "filas_afectadas", "porcentaje_afectado", "accion_aplicada", "filas_silver",
                       "filas_cuarentena", "estado_regla"]].to_string(index=False))
print()
print("Una fila puede activar varias reglas. La suma de filas_afectadas, filas_silver o filas_cuarentena "
      "entre reglas no representa registros unicos y no se usa para conciliar filas.")
print("El reporte de calidad no participa en Bronze = Silver + cuarentena; se valida regla por regla.")

Reporte de calidad: una fila por regla
   regla_id                                                                               columna_evaluada      dimension_calidad   severidad  filas_evaluadas  filas_afectadas  porcentaje_afectado                              accion_aplicada  filas_silver  filas_cuarentena               estado_regla
IOT-CNV-001 temperatura_cabina_c;humedad_cabina_pct;desviacion_termica_flag;desviacion_proximos_60min_flag              exactitud        alta            28920                0             0.000000                              sin_incidencias             0                 0   evaluada_sin_incidencias
IOT-UNI-001                                                                                      temp_unit              exactitud informativa            28920               50             0.001729       conversion_exacta_fahrenheit_a_celsius            49                 1    aplicada_transformacion
IOT-UNI-002                                               

In [7]:
silver_export.to_csv(PATHS["silver"], index=False, encoding="utf-8")
quarantine_export.to_csv(PATHS["quarantine"], index=False, encoding="utf-8")
reporte_calidad.to_csv(PATHS["reporte_calidad"], index=False, encoding="utf-8")

for nombre in ["silver", "quarantine", "reporte_calidad"]:
    ruta = PATHS[nombre]
    print(f"exportado {CONFIG['rutas'][nombre]}: {ruta.stat().st_size} bytes")
print()
copia = PATHS["copia_obsoleta_no_entregable"]
print("copia obsoleta no entregable:", CONFIG["rutas"]["copia_obsoleta_no_entregable"],
      "| existe:", copia.exists(), "| bytes:", copia.stat().st_size if copia.exists() else 0,
      "| leida: False | escrita: False | en conciliacion: False")

exportado datos/silver/andinalog_iot_telemetry_silver.csv: 4766595 bytes
exportado datos/quarantine/andinalog_iot_telemetry_quarantine.csv: 165613 bytes
exportado datos/quality/andinalog_iot_telemetry_reporte_calidad.csv: 10774 bytes

copia obsoleta no entregable: datos/silver/andinalog_iot_telemetry_silver - Copy.csv | existe: True | bytes: 15995602 | leida: False | escrita: False | en conciliacion: False


In [8]:
def recalcular_conteos_independientes():
    """Segunda implementacion de cada regla, leida desde Bronze y los CSV ya exportados."""
    b = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])
    s = pd.read_csv(PATHS["silver"], **CONFIG["lectura"])
    q = pd.read_csv(PATHS["quarantine"], **CONFIG["lectura"])
    en_silver = set(s["_fila_bronze"].astype(int))
    en_cuarentena = set(q["_fila_bronze"].astype(int))
    linea = b.index + 2

    ts = pd.to_datetime(b["timestamp"], format=CONFIG["formato_fecha"]["format"], errors="coerce")
    utc = ts.dt.tz_localize(CONFIG["zonas_horarias"]["sin_zona"], ambiguous="NaT",
                            nonexistent="NaT").dt.tz_convert("UTC")
    temp = pd.to_numeric(b["temperatura_cabina_c"], errors="coerce")
    hum = pd.to_numeric(b["humedad_cabina_pct"], errors="coerce")
    flag1 = pd.to_numeric(b["desviacion_termica_flag"], errors="coerce")
    flag2 = pd.to_numeric(b["desviacion_proximos_60min_flag"], errors="coerce")
    unidad = b["temp_unit"].str.strip().str.upper()
    viaje = b["viaje_id"].str.strip().str.upper()
    orden = b["order_id"].str.strip().str.upper()
    camion = b["camion_id"].str.strip().str.upper()
    producto = b["producto_id"].str.strip().str.upper()
    ident = {columna: b[columna].str.strip().str.upper() for columna in CONFIG["columnas"]["identificadores"]}

    wms = wms_silver.assign(_o=wms_silver["order_id"].str.strip().str.upper()).set_index("_o")
    coincide_wms = orden.isin(wms.index)
    catalogo = productos_silver.assign(_p=productos_silver["producto_id"].str.strip().str.upper()).set_index("_p")
    catalogo = catalogo.assign(
        _req=pd.to_numeric(catalogo["temperatura_conservacion_requerida_c"], errors="coerce"),
        _tol=pd.to_numeric(catalogo["tolerancia_temperatura_c"], errors="coerce"))
    flota = set(flota_silver["camion_id"].str.strip().str.upper())
    pedido_producto = orden.map(wms["producto_id"].str.strip().str.upper())
    pedido_camion = orden.map(wms["camion_id"].str.strip().str.upper())
    pedido_req = producto.map(catalogo["_req"])
    pedido_tol = producto.map(catalogo["_tol"])

    centinela = temp.isin(CONFIG["centinelas"]["temperatura_cabina_c"])
    es_f = unidad.eq("F")
    es_k = unidad.eq("K")
    desconocida = ~unidad.isin(["C", "F", "K"])
    temp_efectiva = temp.mask(centinela)
    temp_efectiva.loc[es_f] = (temp.loc[es_f] - 32) * 5 / 9
    temp_efectiva = temp_efectiva.mask(es_k | desconocida)
    calculada = (temp_efectiva < pedido_req - pedido_tol) | (temp_efectiva > pedido_req + pedido_tol)
    comparable = temp_efectiva.notna() & pedido_req.notna() & pedido_tol.notna()

    clave = pd.DataFrame({"viaje_id": viaje, "timestamp": b["timestamp"].str.strip()})
    duplicada = clave.duplicated(keep=False)
    serie = pd.DataFrame({"viaje_id": viaje, "ts": utc}).sort_values(["viaje_id", "ts"], kind="stable")
    delta = serie.groupby("viaje_id", sort=False)["ts"].diff().dt.total_seconds().div(60)
    fuera_orden = pd.Series(False, index=b.index)
    hueco = pd.Series(False, index=b.index)
    fuera_orden.loc[serie.index] = (delta < 0).to_numpy()
    hueco.loc[serie.index] = (delta > CONFIG["politica_secuencia"]["cadencia_esperada_min"]).to_numpy()
    valido = utc.notna()

    evaluables = {
        "todas": pd.Series(True, index=b.index),
        "timestamp_valido": valido,
        "wms_coincidente": coincide_wms,
        "desviacion_comparable": comparable,
    }
    mascaras = {
        "IOT-CNV-001": ((temp.isna() & b["temperatura_cabina_c"].ne(""))
                        | (hum.isna() & b["humedad_cabina_pct"].ne(""))
                        | (flag1.notna() & ~flag1.isin([0, 1]))
                        | (flag2.notna() & ~flag2.isin([0, 1]))),
        "IOT-UNI-001": es_f,
        "IOT-UNI-002": es_k,
        "IOT-UNI-003": desconocida,
        "IOT-SEN-001": centinela,
        "IOT-SEN-002": hum.isin(CONFIG["centinelas"]["humedad_cabina_pct"]),
        "IOT-NUL-001": b["temperatura_cabina_c"].str.strip().eq(""),
        "IOT-NUL-002": b["humedad_cabina_pct"].str.strip().eq(""),
        "IOT-RNG-001": hum.notna() & ~hum.between(0, 100),
        "IOT-FMT-001": pd.concat(
            [~ident[columna].str.match(patron) for columna, patron in CONFIG["formatos_identificador"].items()],
            axis=1).any(axis=1),
        "IOT-IDN-001": pd.concat(
            [b[columna].ne(ident[columna]) for columna in CONFIG["columnas"]["identificadores"]],
            axis=1).any(axis=1),
        "IOT-FEC-001": ~b["timestamp"].str.match(CONFIG["formato_fecha"]["regex"]),
        "IOT-FEC-002": ~valido,
        "IOT-CLA-001": duplicada,
        "IOT-SEQ-001": fuera_orden,
        "IOT-SEQ-002": hueco,
        "IOT-REF-001": ~coincide_wms,
        "IOT-REF-002": ~producto.isin(set(catalogo.index)),
        "IOT-REF-003": ~camion.isin(flota),
        "IOT-REF-004": coincide_wms & (producto.ne(pedido_producto) | camion.ne(pedido_camion)),
        "IOT-OPE-001": comparable & ~flag1.astype("Float64").eq(calculada.astype("Float64")),
        "IOT-IMP-001": pd.Series(False, index=b.index),
    }
    salida = {}
    for regla in CONFIG["reglas"]:
        if regla.get("no_aplicable"):
            evaluable_regla = pd.Series(False, index=b.index)
        else:
            evaluable_regla = evaluables[regla["evaluable"]]
        efectiva = mascaras[regla["regla_id"]] & evaluable_regla
        salida[regla["regla_id"]] = {
            "filas_evaluadas": int(evaluable_regla.sum()),
            "filas_afectadas": int(efectiva.sum()),
            "filas_silver": int(linea[efectiva & linea.isin(en_silver)].size),
            "filas_cuarentena": int(linea[efectiva & linea.isin(en_cuarentena)].size),
        }
    return salida


reporte_leido = pd.read_csv(PATHS["reporte_calidad"], **CONFIG["lectura"])
recalculado = recalcular_conteos_independientes()
comparacion = []
for regla in CONFIG["reglas"]:
    exportado = reporte_leido.loc[reporte_leido["regla_id"].eq(regla["regla_id"])].iloc[0]
    for metrica, valor in recalculado[regla["regla_id"]].items():
        comparacion.append({
            "regla_id": regla["regla_id"],
            "metrica": metrica,
            "recalculado": valor,
            "reportado": int(exportado[metrica]),
            "coincide": int(exportado[metrica]) == valor,
        })
tabla_comparacion = pd.DataFrame(comparacion)
print("Coincidencia entre el reporte persistido y el recalculo independiente de cada regla")
print(tabla_comparacion.pivot_table(index="regla_id", columns="metrica", values="coincide",
                                    aggfunc="all").to_string())
discrepancias = tabla_comparacion.loc[~tabla_comparacion["coincide"]]
REPORTE_CALIDAD_COINCIDE = discrepancias.empty
assert REPORTE_CALIDAD_COINCIDE, f"conteos del reporte que no coinciden con el recalculo:\n{discrepancias}"
print()
print("REPORTE_CALIDAD_COINCIDE", len(reporte_leido), "reglas verificadas una por una")
print("Reglas con cero incidencias:", reporte_leido.loc[
    reporte_leido["filas_afectadas"].astype(int).eq(0), "regla_id"].tolist())
print("Reglas no aplicables por configuracion:", reporte_leido.loc[
    reporte_leido["estado_regla"].eq("no_aplicable"), "regla_id"].tolist())
print("Reglas con incidencias:", reporte_leido.loc[
    reporte_leido["filas_afectadas"].astype(int).gt(0), "regla_id"].tolist())

Coincidencia entre el reporte persistido y el recalculo independiente de cada regla
metrica      filas_afectadas  filas_cuarentena  filas_evaluadas  filas_silver
regla_id                                                                     
IOT-CLA-001             True              True             True          True
IOT-CNV-001             True              True             True          True
IOT-FEC-001             True              True             True          True
IOT-FEC-002             True              True             True          True
IOT-FMT-001             True              True             True          True
IOT-IDN-001             True              True             True          True
IOT-IMP-001             True              True             True          True
IOT-NUL-001             True              True             True          True
IOT-NUL-002             True              True             True          True
IOT-OPE-001             True              True            

In [9]:
silver_file = pd.read_csv(PATHS["silver"], **CONFIG["lectura"])
quarantine_file = pd.read_csv(PATHS["quarantine"], **CONFIG["lectura"])
reporte_file = pd.read_csv(PATHS["reporte_calidad"], **CONFIG["lectura"])
bronze_relectura = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])

CONTROLES = []


def registrar_control(numero, nombre, resultado, detalle=""):
    CONTROLES.append({"n": numero, "control": nombre, "resultado": bool(resultado), "detalle": detalle})


registrar_control(1, "bronze_sin_modificar",
                  sha256_de_archivo(PATHS["bronze"]) == BRONZE_SHA256_ANTES,
                  f"sha256 identico al inicio de la ejecucion: {BRONZE_SHA256_ANTES}")
registrar_control(2, "conciliacion_bronze_silver_cuarentena",
                  len(bronze_relectura) == len(silver_file) + len(quarantine_file),
                  f"{len(bronze_relectura)} = {len(silver_file)} + {len(quarantine_file)}")
registrar_control(3, "fila_bronze_disjunta",
                  set(silver_file["_fila_bronze"]).isdisjoint(set(quarantine_file["_fila_bronze"])),
                  f"{len(set(silver_file['_fila_bronze']))} claves en Silver y "
                  f"{len(set(quarantine_file['_fila_bronze']))} en cuarentena, sin solape")
registrar_control(4, "fila_bronze_exhaustiva",
                  set(silver_file["_fila_bronze"]) | set(quarantine_file["_fila_bronze"])
                  == {str(linea) for linea in range(2, len(bronze_relectura) + 2)},
                  "la union cubre exactamente las lineas fisicas de Bronze")
registrar_control(5, "silver_sin_errores_bloqueantes",
                  "errores_bloqueantes" not in silver_file.columns
                  and int(reporte_file.loc[reporte_file["estado_regla"].eq("envio_a_cuarentena"),
                                           "filas_silver"].astype(int).sum()) == 0,
                  "Silver no exporta la columna de bloqueantes y cada regla bloqueante declara filas_silver igual a cero")
registrar_control(6, "cuarentena_con_motivo_en_todas_las_filas",
                  bool(quarantine_file["errores_bloqueantes"].ne("").all()
                       and quarantine_file["calidad_motivo"].ne("").all()),
                  f"filas sin motivo: {int(quarantine_file['calidad_motivo'].eq('').sum())}")
registrar_control(7, "clave_lectura_unica_en_silver",
                  not silver_file.duplicated(CONFIG["clave_lectura"]).any(),
                  f"duplicados: {int(silver_file.duplicated(CONFIG['clave_lectura']).sum())}")

duplicadas_en_cuarentena = quarantine_file["errores_bloqueantes"].str.contains("IOT-CLA-001")
claves_duplicadas_bronze = work.loc[work["_lectura_clave_duplicada"], "_fila_bronze"].to_numpy()
filas_duplicadas_bronze = bronze_relectura.iloc[claves_duplicadas_bronze - 2]
ocurrencias_recalculadas = filas_duplicadas_bronze.assign(
    _clave=filas_duplicadas_bronze["viaje_id"].str.strip().str.upper() + " | "
    + filas_duplicadas_bronze["timestamp"].str.strip()).groupby("_clave")["_clave"].transform("size")
registrar_control(8, "duplicados_todas_las_ocurrencias_en_cuarentena",
                  int(duplicadas_en_cuarentena.sum()) == len(claves_duplicadas_bronze)
                  and bool((quarantine_file.loc[duplicadas_en_cuarentena, "ocurrencias_clave_lectura"]
                            .astype(int).to_numpy() == ocurrencias_recalculadas.to_numpy()).all()),
                  f"{int(duplicadas_en_cuarentena.sum())} filas de {work.loc[work['_lectura_clave_duplicada'], '_clave_bronze'].nunique()} claves duplicadas; "
                  f"ocurrencias declaradas coinciden con el recalculo y ninguna quedo en Silver")

ts_silver = pd.to_datetime(silver_file["timestamp"], format="ISO8601", utc=True, errors="coerce")
registrar_control(9, "timestamp_silver_en_utc",
                  bool(silver_file["timestamp"].str.endswith("+00:00").all() and ts_silver.notna().all()),
                  f"sufijo +00:00 en todas las filas; rango {ts_silver.min()} a {ts_silver.max()}")

temp_silver = pd.to_numeric(silver_file["temperatura_cabina_c"], errors="coerce")
es_fahrenheit = silver_file["temp_unit_original"].str.strip().str.upper().eq("F")
f_original = pd.to_numeric(silver_file.loc[es_fahrenheit, "temperatura_cabina_c_original"], errors="coerce")
f_final = pd.to_numeric(silver_file.loc[es_fahrenheit, "temperatura_cabina_c"], errors="coerce")
conversion_exacta = bool(f_original.notna().all()
                         and np.allclose(f_final.to_numpy(), ((f_original - 32) * 5 / 9).to_numpy()))
registrar_control(10, "temperatura_silver_en_celsius",
                  bool(temp_silver.notna().all()
                       and not temp_silver.isin(CONFIG["centinelas"]["temperatura_cabina_c"]).any()
                       and not silver_file["temp_unit_original"].str.strip().str.upper().eq("K").any()
                       and conversion_exacta),
                  f"rango {temp_silver.min()} a {temp_silver.max()}; sin centinelas; las {int(es_fahrenheit.sum())} "
                  f"lecturas convertidas desde Fahrenheit coinciden con la formula exacta")
registrar_control(11, "temp_unit_silver_igual_c",
                  bool(silver_file["temp_unit"].eq(CONFIG["unidades_temperatura"]["canonica"]).all()),
                  f"valores en Silver: {silver_file['temp_unit'].value_counts().to_dict()}")
registrar_control(12, "reporte_de_calidad_coincidente_con_el_recalculo", REPORTE_CALIDAD_COINCIDE,
                  "las 22 reglas se recalcularon de forma independiente desde Bronze y los CSV exportados")
sin_incidencias = reporte_file.loc[reporte_file["estado_regla"].eq("evaluada_sin_incidencias")]
no_aplicables_archivo = reporte_file.loc[reporte_file["estado_regla"].eq("no_aplicable")]
registrar_control(13, "reglas_sin_incidencias_diferenciadas_de_no_evaluadas",
                  bool(reporte_file.loc[reporte_file["estado_regla"].eq("evaluada_sin_incidencias"),
                                        "filas_afectadas"].astype(int).eq(0).all()
                       and sin_incidencias["filas_evaluadas"].astype(int).gt(0).all()
                       and len(no_aplicables_archivo) > 0
                       and no_aplicables_archivo["filas_evaluadas"].astype(int).eq(0).all()
                       and no_aplicables_archivo["accion_aplicada"].ne("sin_incidencias").all()
                       and no_aplicables_archivo["regla_id"].tolist() == ["IOT-IMP-001"]),
                  f"reglas evaluadas sin incidencias con denominador mayor que cero: {len(sin_incidencias)}; "
                  f"regla no aplicable: {no_aplicables_archivo['regla_id'].tolist()} con denominador cero y "
                  f"accion declarada '{no_aplicables_archivo['accion_aplicada'].iloc[0]}'")

for nombre, marco, declaradas in [("silver", silver_file, CONFIG["columnas_silver"]),
                                  ("cuarentena", quarantine_file, CONFIG["columnas_cuarentena"]),
                                  ("reporte_calidad", reporte_file, CONFIG["columnas_reporte_calidad"])]:
    registrar_control(14, f"orden_y_lista_de_columnas_{nombre}", list(marco.columns) == declaradas,
                      f"{len(marco.columns)} columnas en el orden declarado en config")

recibidos = quarantine_file.sort_values("_fila_bronze", key=lambda serie: serie.astype(int)).reset_index(drop=True)
esperados = bronze_relectura.iloc[recibidos["_fila_bronze"].astype(int).to_numpy() - 2].reset_index(drop=True)
coinciden_recibidos = all(
    esperados[columna].reset_index(drop=True).equals(recibidos[f"{columna}_original"])
    for columna in CONFIG["columnas"]["obligatorias"]
)
registrar_control(15, "valores_recibidos_de_cuarentena_coinciden_con_bronze", coinciden_recibidos,
                  "los valores originales de cuarentena se recuperan por _fila_bronze y son identicos a Bronze")

constantes_silver = [columna for columna in silver_file.columns if silver_file[columna].nunique(dropna=False) <= 1]
constantes_cuarentena = [columna for columna in quarantine_file.columns
                         if quarantine_file[columna].nunique(dropna=False) <= 1]
sin_documentar = ([columna for columna in constantes_silver
                   if columna not in CONFIG["columnas_constantes_documentadas"]["silver"]]
                  + [columna for columna in constantes_cuarentena
                     if columna not in CONFIG["columnas_constantes_documentadas"]["quarantine"]])
registrar_control(16, "sin_columnas_constantes_no_documentadas", not sin_documentar,
                  f"constantes en Silver {constantes_silver}; constantes en cuarentena {constantes_cuarentena}; "
                  f"todas con funcion documentada en config")

repetidas = []
for nombre, marco in [("silver", silver_file), ("quarantine", quarantine_file)]:
    columnas = list(marco.columns)
    for indice, izquierda in enumerate(columnas):
        for derecha in columnas[indice + 1:]:
            if marco[izquierda].equals(marco[derecha]):
                repetidas.append(f"{nombre}: {izquierda} es identica a {derecha}")
registrar_control(17, "ausencia_de_columnas_duplicadas", not repetidas,
                  f"pares con contenido identico: {repetidas if repetidas else 'ninguno'}")

faltantes_gold = [columna for columna in CONFIG["columnas_requeridas_por_gold"] if columna not in silver_file.columns]
registrar_control(18, "compatibilidad_con_las_columnas_requeridas_por_gold", not faltantes_gold,
                  f"faltantes: {faltantes_gold if faltantes_gold else 'ninguna'}; Gold lee "
                  f"{CONFIG['columnas_requeridas_por_gold_opcionales']} con comprobacion opcional, "
                  f"por lo que no se exigen columnas vacias")

registrar_control(19, "copia_obsoleta_excluida_de_entradas_y_entregables",
                  CONFIG["rutas"]["copia_obsoleta_no_entregable"] not in CONFIG["rutas_de_entrada"]
                  and CONFIG["rutas"]["copia_obsoleta_no_entregable"] not in CONFIG["rutas_de_salida"]
                  and all("copy" not in CONFIG["rutas"][nombre].lower()
                          for nombre in CONFIG["rutas_de_entrada"] + CONFIG["rutas_de_salida"])
                  and (not COPIA_OBSOLETA_EXISTE
                       or sha256_de_archivo(PATHS["copia_obsoleta_no_entregable"])
                       == COPIA_OBSOLETA_SHA256_ANTES),
                  f"{CONFIG['rutas']['copia_obsoleta_no_entregable']} no se lee, no se escribe, no se concilia "
                  f"y no se entrega; sigue intacta con {COPIA_OBSOLETA_BYTES} bytes y el mismo sha256")

registrar_control(20, "ninguna_fila_se_descarta_en_el_pipeline",
                  len(work) == len(bronze_relectura)
                  and len(silver_file) + len(quarantine_file) == len(bronze_relectura)
                  and not work["_fila_bronze"].duplicated().any(),
                  f"el dataframe de trabajo conserva las {len(work)} filas de Bronze y las dos proyecciones las "
                  f"reparten sin eliminar ni agregar registros; ningun error se descarta, se enruta")


def cerrar_hallazgo_banderas_informativas():
    """Control de cierre del hallazgo H-1 de la auditoria: la cuarentena conserva sus banderas informativas."""
    n = lambda serie: serie.str.strip().str.upper()
    q = quarantine_file.reset_index(drop=True)
    informative = {regla["regla_id"] for regla in CONFIG["reglas"] if regla["informativa"]}
    tokens_por_fila = q["banderas_informativas"].map(
        lambda valor: [token.strip() for token in valor.split("|") if token.strip()])
    todos_los_tokens = {token for tokens in tokens_por_fila for token in tokens}
    conteos_coinciden = bool((tokens_por_fila.map(len).astype(int)
                              == q["conteo_banderas_informativas"].astype(int)).all())
    sin_conteo_sin_token = int((q["conteo_banderas_informativas"].astype(int).gt(0)
                                & q["banderas_informativas"].eq("")).sum())
    filas_con_informativo = int(q["conteo_banderas_informativas"].astype(int).gt(0).sum())
    lineas = q["_fila_bronze"].astype(int).to_numpy()
    recibido = bronze_relectura.iloc[lineas - 2].reset_index(drop=True)
    a_lineas = lambda mascara: {int(lineas[posicion]) for posicion in np.flatnonzero(mascara)}
    esperado = {
        "IOT-REF-001": a_lineas(~n(recibido["order_id"]).isin(set(n(wms_silver["order_id"]))).to_numpy()),
        "IOT-REF-002": a_lineas(~n(recibido["producto_id"]).isin(set(n(productos_silver["producto_id"]))).to_numpy()),
        "IOT-REF-003": a_lineas(~n(recibido["camion_id"]).isin(set(n(flota_silver["camion_id"]))).to_numpy()),
        "IOT-NUL-002": a_lineas(recibido["humedad_cabina_pct"].str.strip().eq("").to_numpy()),
        "IOT-SEQ-002": {int(quarantine_file.loc[posicion, "_fila_bronze"])
                        for posicion in np.flatnonzero(MARCAS["IOT-SEQ-002"][quarantine.index])},
    }
    conjuntos = {regla_id: {int(lineas[posicion]) for posicion, tokens in enumerate(tokens_por_fila)
                            if regla_id in tokens} for regla_id in sorted(informative)}
    conjuntos_ok = all(conjuntos[regla_id] == esperado[regla_id] for regla_id in sorted(informative))
    return {
        "columna_presente": "banderas_informativas" in quarantine_file.columns,
        "tokens_conocidos": todos_los_tokens <= informative,
        "conteos_coinciden": conteos_coinciden,
        "sin_conteo_sin_token": sin_conteo_sin_token,
        "filas_con_informativo": filas_con_informativo,
        "conjuntos": {regla_id: len(valores) for regla_id, valores in sorted(conjuntos.items())},
        "conjuntos_ok": conjuntos_ok,
    }


cierre_h1 = cerrar_hallazgo_banderas_informativas()
registrar_control(25, "cierre_hallazgo_H1_banderas_informativas_en_cuarentena",
                  cierre_h1["columna_presente"] and cierre_h1["tokens_conocidos"]
                  and cierre_h1["conteos_coinciden"] and cierre_h1["sin_conteo_sin_token"] == 0
                  and cierre_h1["conjuntos_ok"],
                  f"columna presente {cierre_h1['columna_presente']}; tokens conocidos {cierre_h1['tokens_conocidos']}; "
                  f"numero de tokens igual al conteo en las {len(quarantine_file)} filas {cierre_h1['conteos_coinciden']}; "
                  f"filas con conteo informativo mayor que cero y token vacio {cierre_h1['sin_conteo_sin_token']}; "
                  f"filas con informativa {cierre_h1['filas_con_informativo']}; conjuntos por regla recalculados "
                  f"desde Bronze y las fuentes maestras coinciden {cierre_h1['conjuntos_ok']} {cierre_h1['conjuntos']}")
registrar_control(26, "cierre_hallazgo_H5_regla_no_aplicable",
                  bool(len(no_aplicables_archivo) == 1
                       and no_aplicables_archivo["regla_id"].iloc[0] == "IOT-IMP-001"
                       and int(no_aplicables_archivo["filas_evaluadas"].iloc[0]) == 0
                       and no_aplicables_archivo["accion_aplicada"].iloc[0] == "no_imputada_por_config"),
                  f"IOT-IMP-001 queda como no aplicable con denominador 0 y accion "
                  f"'{no_aplicables_archivo['accion_aplicada'].iloc[0]}'")
registrar_control(27, "cierre_hallazgo_H4_evidencia_de_duplicados",
                  bool(str(claves_duplicadas) in reporte_file.loc[reporte_file["regla_id"].eq("IOT-CLA-001"),
                                                                 "evidencia"].iloc[0]
                       and str(exacto_en_clave_duplicada) in reporte_file.loc[
                           reporte_file["regla_id"].eq("IOT-CLA-001"), "evidencia"].iloc[0]
                       and str(claves_con_diferencias) in reporte_file.loc[
                           reporte_file["regla_id"].eq("IOT-CLA-001"), "evidencia"].iloc[0]),
                  f"la evidencia de IOT-CLA-001 declara {claves_duplicadas} claves, {filas_duplicadas} filas, "
                  f"{exacto_en_clave_duplicada} duplicados exactos y {claves_con_diferencias} claves con "
                  f"diferencias en {filas_en_claves_con_diferencias} filas")
registrar_control(28, "cierre_hallazgo_H6_evidencia_sin_anglicismos",
                  not reporte_file["evidencia"].str.contains("received").any(),
                  "ninguna evidencia del reporte de calidad contiene el anglicismo 'received'")

tabla_controles = pd.DataFrame(CONTROLES)
print("Controles finales ejecutados sobre los tres CSV persistidos")
print(tabla_controles[["n", "control", "resultado", "detalle"]].to_string(index=False))
fallidos = tabla_controles.loc[~tabla_controles["resultado"], "control"].tolist()
assert not fallidos, f"controles fallidos: {fallidos}"
print()
print("CONTROLES_OK", len(tabla_controles), "controles, todos True")

Controles finales ejecutados sobre los tres CSV persistidos
 n                                                control  resultado                                                                                                                                                                                                                                                                                                                                                                        detalle
 1                                   bronze_sin_modificar       True                                                                                                                                                                                                                                                                    sha256 identico al inicio de la ejecucion: edb7afe2f7fb836e59fe605d30c88b3b5b13a6d8ab2ec0b37f206a14e58de6bf
 2                  conciliacion_bronze_silver_cuarentena       True

In [10]:
FUNCIONES_SILVER = [
    "marca de tiempo canonica en UTC",
    "entidad de la lectura",
    "contexto de la orden",
    "contexto del camion",
    "contexto del producto",
    "medida de temperatura en Celsius",
    "unidad canonica de la temperatura",
    "humedad relativa; vacia si estaba ausente en el valor recibido",
    "hecho de desviacion termica observado en el instante t",
    "etiqueta de desviacion en los proximos 60 minutos entregada por Bronze",
    "trazabilidad a la linea fisica de Bronze",
    "timestamp recibido; demuestra la conversion de zona horaria",
    "unidad recibida; demuestra el origen Fahrenheit de la conversion",
    "temperatura recibida; demuestra la conversion y la eliminacion de centinelas",
    "motivos de transformacion aplicados a la fila",
    "conteo resumido de transformaciones aplicadas",
    "banderas informativas que no bloquean la fila",
    "estado de calidad; en Silver solo puede ser valida",
]
FUNCIONES_CUARENTENA = [
    "posicion fisica de la fila en Bronze",
    "regla_id de cada error bloqueante, separados por pipe",
    "descripcion legible del error bloqueante principal",
    "estado de calidad; en cuarentena siempre es cuarentena",
    "clave de lectura tal como se recibio en Bronze",
    "clave de lectura con identificador normalizado y timestamp en UTC",
    "timestamp recibido",
    "viaje_id recibido",
    "order_id recibido",
    "camion_id recibido",
    "producto_id recibido",
    "temperatura recibida, centinelas incluidos",
    "unidad recibida",
    "humedad recibida",
    "desviacion_termica_flag recibida",
    "desviacion_proximos_60min_flag recibida",
    "instante al que se llega con la normalizacion; vacio si la fecha es imposible",
    "temperatura ya normalizada a Celsius y sin centinela; vacia si no es corregible",
    "control de duplicidad; habilita la accion posterior de recuperacion",
    "numero de ocurrencias de la clave de lectura en Bronze",
    "control referencial: existe correspondencia en WMS Silver",
    "control referencial: existe correspondencia en Productos Silver",
    "control referencial: existe correspondencia en Flota Silver",
    "motivos de transformacion aplicados antes del enrutamiento",
    "motivos de imputacion; vacio porque la imputacion esta deshabilitada en config",
    "banderas informativas que no bloquean la fila, identificadas por regla_id",
    "conteo de transformaciones aplicadas",
    "conteo de errores bloqueantes de la fila",
    "conteo de banderas informativas de la fila",
]
perfil_columnas = pd.DataFrame({"columna_silver": CONFIG["columnas_silver"], "funcion": FUNCIONES_SILVER})
perfil_cuarentena = pd.DataFrame({"columna_cuarentena": CONFIG["columnas_cuarentena"],
                                  "funcion": FUNCIONES_CUARENTENA})
criterio_eliminacion = pd.DataFrame({
    "grupo": [
        "alias _tratado",
        "copias _original que nunca cambian",
        "banderas bloqueantes siempre falsas en Silver",
        "columnas constantes",
        "helpers internos de calculo",
    ],
    "columnas retiradas": [
        "timestamp_tratado, viaje_id_tratado, order_id_tratado, camion_id_tratado, producto_id_tratado, "
        "temp_unit_tratado, temperatura_cabina_c_tratado, humedad_cabina_pct_tratado, flags _tratado",
        "viaje_id_original, order_id_original, producto_id_original, humedad_cabina_pct_original, "
        "desviacion_termica_flag_original, desviacion_proximos_60min_flag_original, camion_id_original",
        "temperatura_cabina_c_centinela_detectado, temperatura_cabina_c_conversion_invalida, "
        "temperatura_unidad_no_esperada, temperatura_unidad_desconocida, temperatura_cabina_c_ausente, "
        "humedad_cabina_pct_fuera_rango, timestamp_conversion_invalida, timestamp_formato_reconocido, "
        "lectura_clave_duplicada, order_id_corresponde_wms_silver, producto_id_corresponde_silver, "
        "camion_id_corresponde_silver, producto_id_coherente_con_wms, camion_id_coherente_con_wms, "
        "desviacion_termica_calculable, desviacion_termica_calculada, desviacion_termica_flag_coherente",
        "errores_bloqueantes, fue_imputada, imputacion_metodo, imputacion_motivo, conteo_imputaciones, "
        "calidad_motivo, motivos_imputacion, temperatura_convertida_fahrenheit",
        "t_viaje_id, t_order_id, t_camion_id, t_producto_id, t_temp_unit, t_temperatura_c, t_humedad_pct, "
        "t_flag_actual, t_flag_futuro, _ocurrencias, _clave_bronze, _corresponde_* y las marcas de cada regla",
    ],
    "justificacion": [
        "alias del valor canonico ya presente en la columna final",
        "no demuestran ninguna transformacion; la normalizacion reversible de identificadores queda auditada "
        "por IOT-IDN-001 y es recuperable con _fila_bronze",
        "toda regla bloqueante debe declarar filas_silver igual a cero en el reporte de calidad, por lo que "
        "exportarlas en Silver no aporta informacion de consumo",
        "sin variacion ni funcion de consumo; la ausencia de imputaciones se demuestra con la regla IOT-IMP-001",
        "marcas de regla y valores intermedios: no son parte de ningun entregable",
    ],
})
motivos_cuarentena = {}
for valor in quarantine_file["errores_bloqueantes"]:
    for motivo in [p.strip() for p in valor.split("|") if p.strip()]:
        motivos_cuarentena[motivo] = motivos_cuarentena.get(motivo, 0) + 1
motivos_cuarentena = dict(sorted(motivos_cuarentena.items(), key=lambda par: (-par[1], par[0])))
informativas_silver = {}
for valor in silver_file["banderas_informativas"]:
    for motivo in [p.strip() for p in valor.split("|") if p.strip()]:
        informativas_silver[motivo] = informativas_silver.get(motivo, 0) + 1
informativas_silver = dict(sorted(informativas_silver.items(), key=lambda par: (-par[1], par[0])))
informativas_cuarentena = {}
for valor in quarantine_file["banderas_informativas"]:
    for motivo in [p.strip() for p in valor.split("|") if p.strip()]:
        informativas_cuarentena[motivo] = informativas_cuarentena.get(motivo, 0) + 1
informativas_cuarentena = dict(sorted(informativas_cuarentena.items(), key=lambda par: (-par[1], par[0])))
reglas_por_estado = {}
for fila in reporte_file.itertuples():
    reglas_por_estado.setdefault(fila.estado_regla, []).append(fila.regla_id)
reglas_por_estado = {estado: reglas for estado, reglas in sorted(reglas_por_estado.items())}
conteo_reglas = quarantine_file["errores_bloqueantes"].map(
    lambda valor: len([p for p in valor.split("|") if p.strip()]))
transformaciones_silver = {}
for valor in silver_file["motivos_transformacion"]:
    for motivo in [p.strip() for p in valor.split("|") if p.strip()]:
        transformaciones_silver[motivo] = transformaciones_silver.get(motivo, 0) + 1
transformaciones_silver = dict(sorted(transformaciones_silver.items(), key=lambda par: (-par[1], par[0])))

informe = f"""# Informe B2S 04 - AndinaLog IoT Telemetry

## Objetivo, entidad y granularidad

Conversion auditada de lecturas IoT desde Bronze hacia un Silver compacto, una cuarentena investigable y
un reporte de calidad con una fila por regla.

- Entidad: {CONFIG["entidad"]}.
- Granularidad: {CONFIG["granularidad"]}.
- Clave de lectura: {CONFIG["clave_lectura"]}, unica en Silver.
- Entradas: {CONFIG["rutas"]["bronze"]}, {CONFIG["rutas"]["wms_silver"]},
  {CONFIG["rutas"]["productos_silver"]}, {CONFIG["rutas"]["flota_silver"]}.
- Las fuentes Silver de apoyo se consultan solo para correspondencia y coherencia, y no se modifican.
- No se realizo ninguna imputacion ni se uso informacion futura.

## Perfil Bronze obtenido del archivo real

- Filas: {PERFIL["filas"]}; columnas: {PERFIL["columnas"]}.
- Columnas recibidas: {PERFIL["nombres_columnas"]}.
- Tipos recibidos: {PERFIL["tipos_recibidos"]}.
- Vacios por columna: {PERFIL["vacios"]}.
- Unidades de temperatura observadas: {PERFIL["unidades_observadas"]}.
- Centinelas -999: {PERFIL["centinelas"]}.
- Formato de timestamp reconocido en {PERFIL["fechas_formato_reconocido"]} filas; timestamps imposibles:
  {PERFIL["fechas_invalidas"]} con valores {PERFIL["valores_fecha_invalida"]}.
- Lecturas por viaje: {PERFIL["lecturas_por_viaje"]}.
- Cadencia de la serie normalizada y ordenada: {PERFIL["frecuencia_minutos_ordenada"]} minutos.
- Inversiones del orden fisico del CSV: {PERFIL["inversiones_orden_fisico_archivo"]}.
  Inversiones cronologicas tras ordenar por viaje_id y timestamp: {PERFIL["inversiones_tras_ordenar_por_viaje"]}.
- Duplicados: {PERFIL["duplicados_exactos_filas"]} filas exactas y {PERFIL["duplicados_lectura_filas"]} filas
  en {PERFIL["duplicados_lectura_claves"]} claves de lectura; de esas, {PERFIL["duplicados_no_exactos_filas"]}
  filas tienen diferencias entre si dentro de su clave.
- Rangos numericos recibidos: {PERFIL["rangos_numericos_raw"]}.
- sha256 de Bronze al inicio de la ejecucion: {PERFIL["sha256_bronze_antes"]}.

## Criterio de seleccion de columnas de Silver

Silver no exporta el dataframe de trabajo. `config` declara `columnas_silver`, `columnas_cuarentena` y
`columnas_reporte_calidad`, y el notebook falla si alguna columna declarada no se produce, si el orden
difiere o si hay columnas repetidas.

Silver queda con {len(silver_file.columns)} columnas, todas con funcion verificable:

{tabla_markdown(perfil_columnas)}

Columnas retiradas y justificacion:

{tabla_markdown(criterio_eliminacion)}

Originales conservados solo cuando demuestran una transformacion real: `timestamp_original` difiere del
valor final en las {len(silver_file)} filas por la conversion de Bolivia a UTC, y
`temperatura_cabina_c_original` difiere exactamente en las lecturas convertidas desde Fahrenheit
({int(silver_file["motivos_transformacion"].str.contains("temperatura_fahrenheit_convertida_celsius").sum())} filas).
Los identificadores, la humedad y las banderas no llevan original en Silver porque no cambian. La
posicion exacta en Bronze se recupera siempre con `_fila_bronze`.

Silver y cuarentena no exportan las banderas booleanas `fue_transformada` ni `fue_imputada` porque serian
constantes y no aportarian informacion. La equivalencia verificable es la siguiente:
`fue_transformada` equivale a `conteo_transformaciones > 0` y a `motivos_transformacion` no vacio, y
`fue_imputada` equivale a `motivos_imputacion` no vacio. En esta ejecucion hay
{int(silver_file["motivos_transformacion"].ne("").sum())} de {len(silver_file)} filas de Silver con
transformacion y {int(quarantine_file["motivos_transformacion"].ne("").sum())} de
{len(quarantine_file)} filas de cuarentena con transformacion. No se imputo ninguna fila:
`motivos_imputacion` esta vacio en las {int(quarantine_file["motivos_imputacion"].eq("").sum())} de
{len(quarantine_file)} filas de cuarentena, coherente con la regla IOT-IMP-001, y Silver no exporta esa
columna porque no contiene ninguna imputacion.

Motivos de transformacion en Silver: {transformaciones_silver}.
Banderas informativas en Silver: {informativas_silver}.
Los tokens de `errores_bloqueantes` y de `banderas_informativas` son `regla_id` estables; su
significado esta en `descripcion_regla` del reporte de calidad.

## Valores canonicos de Silver

- `timestamp` es UTC con sufijo `+00:00`, entre {ts_silver.min()} y {ts_silver.max()}.
- `temperatura_cabina_c` es Celsius, entre {temp_silver.min()} y {temp_silver.max()}, sin centinelas.
- `temp_unit` es `C` en las {len(silver_file)} filas.
- `calidad_estado` es `valida` en todas las filas porque Silver no admite errores bloqueantes.

## Cuarentena investigable

Cuarentena queda con {len(quarantine_file.columns)} columnas:

{tabla_markdown(perfil_cuarentena)}

Los errores se identifican con `regla_id` dentro de `errores_bloqueantes` en lugar de una columna booleana
por regla, y lo mismo ocurre con `banderas_informativas`. Se conserva `lectura_clave_duplicada` porque
habilita la accion posterior de recuperacion y evita ambiguedad con `ocurrencias_clave_lectura`. El conteo
de banderas informativas siempre acompaña a los tokens y coincide con ellos fila a fila, verificado por un
control de cierre que recalcula los conjuntos de filas por regla informativa desde Bronze y desde las
fuentes Silver maestras.

- Motivos de cuarentena: {motivos_cuarentena}.
- Banderas informativas en cuarentena: {informativas_cuarentena}; el conteo por fila coincide con el numero
  de tokens en las {len(quarantine_file)} filas.
- Filas con una sola regla: {int(conteo_reglas.eq(1).sum())}.
- Filas con dos reglas: {int(conteo_reglas.eq(2).sum())}; son lecturas con la temperatura ausente cuya
  clave tambien esta duplicada.
- Filas sin motivo: {int(quarantine_file["calidad_motivo"].eq("").sum())}.
- Las activaciones se solapan: la suma de los motivos no es el numero de filas.

## Reglas, duplicados y secuencia temporal

- Fahrenheit convertido a Celsius con la formula exacta {CONFIG["conversion_fahrenheit_celsius"]}:
  {int(MARCAS["IOT-UNI-001"].sum())} lecturas, de las cuales
  {int(silver_file["motivos_transformacion"].str.contains("temperatura_fahrenheit_convertida_celsius").sum())}
  llegan a Silver. El rango Fahrenheit recibido es {rango_por_unidad.get("F")} y el Celsius resultante es
  canonico.
- Kelvin no esperado operacionalmente: {int(MARCAS["IOT-UNI-002"].sum())} lecturas a cuarentena, con rango
  recibido {rango_por_unidad.get("K")}.
- Centinelas -999: {int(MARCAS["IOT-SEN-001"].sum())} en temperatura y
  {int(MARCAS["IOT-SEN-002"].sum())} en humedad. Ninguno es una medicion.
- Temperatura ausente: {int(MARCAS["IOT-NUL-001"].sum())} lecturas a cuarentena.
- Humedad ausente: {int(MARCAS["IOT-NUL-002"].sum())} lecturas, conservadas nulas y reportadas como
  informativa, sin imputar.
- Humedad fuera de 0-100: {int(MARCAS["IOT-RNG-001"].sum())} lecturas a cuarentena, con valores
  {PERFIL["humedad_fuera_de_rango_detallada"]}.
- Timestamp invalido: {int(MARCAS["IOT-FEC-002"].sum())} lecturas a cuarentena.
- Duplicados: politica conservadora declarada en `config`. Las {claves_duplicadas} claves duplicadas y sus
  {filas_duplicadas} ocurrencias van completas a cuarentena con el motivo
  `{CONFIG["politica_duplicados"]["motivo"]}`. No se elige ganador porque no existe regla de negocio
  inequivoca. De esas {filas_duplicadas} filas, {exacto_en_clave_duplicada} son duplicado exacto de su
  clave y {claves_con_diferencias} claves tienen ocurrencias con valores distintos entre si, en
  {filas_en_claves_con_diferencias} filas. Los duplicados exactos no se cuentan como un problema
  adicional independiente.
- Secuencia: `IOT-SEQ-001` se evalua despues de normalizar el timestamp y ordenar por `viaje_id` y
  `timestamp`, y arroja {int(MARCAS["IOT-SEQ-001"].sum())} incidencias. Las
  {PERFIL["inversiones_orden_fisico_archivo"]} inversiones del orden fisico del CSV se documentan como
  caracteristica del archivo de entrada y no se marcan como error de las lecturas. `_fila_bronze`
  conserva la posicion fisica original.
- Cadencia: `IOT-SEQ-002` registra los intervalos mayores que
  {CONFIG["politica_secuencia"]["cadencia_esperada_min"]} minutos como huecos informativos
  ({int(MARCAS["IOT-SEQ-002"].sum())} intervalos de 60 minutos). Ninguna fila va a cuarentena por el
  orden fisico del CSV ni por un intervalo de 60 minutos.
- No se aplican limites termicos inventados: una desviacion operacional valida no es error de calidad.
  Desviaciones termicas observadas: {desviaciones_operacionales}.

## Imputacion

No se imputo ninguna variable. `config` deja las imputaciones deshabilitadas y prohibe
{CONFIG["imputaciones"]["metodos_prohibidos"]}. Motivo declarado: {CONFIG["imputaciones"]["motivo"]}.
La regla `IOT-IMP-001` del reporte se declara no aplicable porque no hay ninguna regla de imputacion en
`config` que aplicar: por eso no tiene denominador de evaluacion, sus conteos son cero y su accion
declarada es `no_imputada_por_config`, distinto de una regla evaluada sin incidencias. La ausencia se
conserva y se reporta, nunca se rellena.

## Integridad referencial

- Lecturas sin correspondencia en WMS Silver: {int(MARCAS["IOT-REF-001"].sum())} en {sin_wms_unicos} order_id normalizados distintos.
- Lecturas sin correspondencia en Productos Silver: {int(MARCAS["IOT-REF-002"].sum())} en {sin_producto_unicos} producto_id normalizados distintos.
- Lecturas sin correspondencia en Flota Silver: {int(MARCAS["IOT-REF-003"].sum())} en {sin_camion_unicos} camion_id normalizados distintos. Los identificadores se comparan ya normalizados, de modo que
  distintas grafias del mismo camion no se cuentan como claves diferentes.
- Contradicciones con una orden WMS existente: {int(MARCAS["IOT-REF-004"].sum())} sobre
  {wms_coincidentes} lecturas con orden coincidente.
- Coherencia de la desviacion termica: {int(MARCAS["IOT-OPE-001"].sum())} incoherencias sobre
  {comparables} lecturas comparables.
- Criterio: {CONFIG["integridad_referencial"]["criterio"]}.

## Reporte de calidad

Archivo: `{CONFIG["rutas"]["reporte_calidad"]}`, una fila por regla y las columnas declaradas en
`columnas_reporte_calidad`.

- Reglas evaluadas: {len(reporte_file)}.
- Reglas con incidencias: {sorted(reglas_por_estado.get("envio_a_cuarentena", []) + reglas_por_estado.get("informativa_sin_cuarentena", []) + reglas_por_estado.get("aplicada_transformacion", []))}.
- Estados presentes: {list(reglas_por_estado)}.
- Reglas evaluadas sin incidencias: {reglas_por_estado.get("evaluada_sin_incidencias", [])}.
- Reglas no aplicables por configuracion: {reglas_por_estado.get("no_aplicable", [])}.

{tabla_markdown(reporte_file[["regla_id", "columna_evaluada", "dimension_calidad", "severidad", "filas_evaluadas", "filas_afectadas", "porcentaje_afectado", "accion_aplicada", "filas_silver", "filas_cuarentena", "estado_regla"]])}

Una fila puede activar varias reglas. La suma de `filas_afectadas`, `filas_silver` o `filas_cuarentena`
entre reglas no representa registros unicos. El reporte de calidad no participa en
`Bronze = Silver + cuarentena`; se valida regla por regla contra un recalculo independiente hecho desde
Bronze y los CSV exportados, y las {len(reporte_file)} reglas coinciden.

## Enrutamiento y resultado

- Estados: {work["calidad_estado"].value_counts().to_dict()}.
- `calidad_estado` se calcula despues del tratamiento y de la imputacion: `imputar` no modifica ninguna
  fila y `asignar_calidad` decide el estado a partir de las reglas bloqueantes.
- Antes de exportar se comprueba que ninguna regla bloqueante este activa en las filas destinadas a
  Silver, y la comprobacion pasa.

## Resultado y conciliacion

- Conciliacion: Bronze {len(bronze_relectura)} = Silver {len(silver_file)} + cuarentena {len(quarantine_file)}.
- Las claves `_fila_bronze` de Silver y cuarentena son disjuntas y su union es exactamente el conjunto de
  lineas de Bronze.
- Silver tiene clave de lectura unica, timestamp en UTC, temperatura en Celsius, `temp_unit` igual a `C`
  y cero errores bloqueantes.
- Los valores recibidos de la cuarentena coinciden fila a fila con Bronze.
- Bronze no fue modificado: mismo sha256 al inicio y al final de la ejecucion.

## Archivos generados

- `{CONFIG["rutas"]["notebook"]}`
- `{CONFIG["rutas"]["silver"]}`
- `{CONFIG["rutas"]["quarantine"]}`
- `{CONFIG["rutas"]["reporte_calidad"]}`
- `{CONFIG["rutas"]["informe"]}`

No leido, no modificado y no entregable: `{CONFIG["rutas"]["copia_obsoleta_no_entregable"]}`. Es una
copia obsoleta de una version anterior de Silver, queda fuera de entradas, busquedas automaticas,
conciliaciones y controles de existencia. Se decidira por separado si se elimina o se mueve fuera de
`datos/silver` despues de la auditoria.

## Limitaciones

- No hay contrato que sostente limites termicos, por lo que no se aplican y no se puede afirmar que una
  temperatura alta o baja sea un error.
- La desviacion termica solo se puede verificar cuando el producto tiene temperatura de conservacion y
  tolerancia en Productos Silver; en las demas lecturas no es evaluable y no se cuenta como incoherencia.
- La humedad ausente y la humedad fuera de rango reciben tratamientos distintos: la primera se conserva
  porque el sensor no entrego lectura, la segunda se rechaza porque el valor recibido es imposible.
- El orden fisico del CSV de Bronze no es una secuencia cronologica y no se usa como tal.
- La etiqueta `desviacion_proximos_60min_flag` se entrega tal como llega de Bronze; esta etapa no la
  recalcula ni la valida contra ventanas temporales.
- El modelo predictivo posterior debe derivar sus ventanas temporales por viaje y no puede usar ninguna
  lectura posterior al instante de prediccion.

## Correcciones aplicadas por la auditoria

La auditoria independiente de esta entrega confirmo un hallazgo importante y cinco hallazgos menores. Este
Build los corrige sin cambiar ninguna decision metodologica aprobada.

- H-1 IMPORTANTE, cerrado: la cuarentena no exportaba `banderas_informativas` aunque si exportaba su
  conteo, de modo que {int(quarantine_file["conteo_banderas_informativas"].astype(int).gt(0).sum())} filas
  tenian un motivo informativo no legible. Se adiciono la columna a `columnas_cuarentena` y a
  `proyectar_cuarentena`; la cuarentena queda con {len(quarantine_file.columns)} columnas y sus
  banderas informativas son {informativas_cuarentena}. El control de cierre verifica que la columna
  existe, que todos sus tokens son `regla_id` declarados, que el numero de tokens coincide con
  `conteo_banderas_informativas` en las {len(quarantine_file)} filas y que los conjuntos de filas por
  regla informativa recalculados desde Bronze y desde las fuentes maestras coinciden con los del archivo.
- H-2 MENOR, cerrado: se documenta en la seccion de seleccion de columnas la equivalencia entre
  `fue_transformada` y `conteo_transformaciones` mas `motivos_transformacion`, y entre `fue_imputada` y
  `motivos_imputacion`, con los conteos de esta ejecucion.
- H-3 MENOR, sin cambio por decision aprobada: `calidad_motivo` no se exporta en Silver porque seria
  constante; la evidencia de cero errores bloqueantes queda en el reporte y en la conciliacion.
- H-4 MENOR, cerrado: la evidencia de `IOT-CLA-001` declara ahora las {claves_duplicadas} claves, las
  {filas_duplicadas} filas, los {exacto_en_clave_duplicada} duplicados exactos y las
  {claves_con_diferencias} claves con diferencias, en {filas_en_claves_con_diferencias} filas.
- H-5 MENOR, cerrado: `IOT-IMP-001` queda con estado `no_aplicable`, denominador cero y accion
  `no_imputada_por_config`.
- H-6 MENOR, cerrado: se corrigieron los anglicismos en los textos de evidencia del reporte de calidad.

## Contradicciones respecto del plan

""" + "\n".join(f"- {item}." for item in CONFIG["contradicciones_plan"]) + f"""

## Reproducibilidad

- Fecha de ejecucion UTC: {EXECUTED_AT_UTC}.
- Python: {platform.python_version()}.
- pandas: {pd.__version__}.
- numpy: {np.__version__}.
- Semilla: {CONFIG["semilla"]} ({CONFIG["nota_semilla"]}).
- Rutas de entrada: {CONFIG["rutas"]["bronze"]}, {CONFIG["rutas"]["wms_silver"]},
  {CONFIG["rutas"]["productos_silver"]}, {CONFIG["rutas"]["flota_silver"]}.
- Rutas de salida: {CONFIG["rutas"]["silver"]}, {CONFIG["rutas"]["quarantine"]},
  {CONFIG["rutas"]["reporte_calidad"]}, {CONFIG["rutas"]["informe"]}.
- Zona inicial de los timestamps sin zona: {CONFIG["zonas_horarias"]["sin_zona"]}; zona canonica de
  Silver: {CONFIG["zonas_horarias"]["silver"]}.
- Conversion de Fahrenheit: {CONFIG["conversion_fahrenheit_celsius"]}.
- Cadencia esperada: {CONFIG["politica_secuencia"]["cadencia_esperada_min"]} minutos.
- Conteos de esta ejecucion: Bronze {len(bronze_relectura)}, Silver {len(silver_file)},
  cuarentena {len(quarantine_file)}, reglas de calidad {len(reporte_file)}.
- sha256 de Bronze: {PERFIL["sha256_bronze_antes"]}.
- Los controles finales se aplican sobre los tres CSV persistidos, no solo sobre los dataframes en
  memoria, y el informe se escribe al final de esa misma ejecucion.
"""

PATHS["informe"].write_text(informe, encoding="utf-8")
print("Informe generado en:", PATHS["informe"])
print("bytes:", PATHS["informe"].stat().st_size)

Informe generado en: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_04_IoT_Telemetry.md
bytes: 36671


In [11]:
import re

texto_informe = PATHS["informe"].read_text(encoding="utf-8")
patron_gold = r"Conciliacion: Bronze (\d+) = Silver (\d+) \+ cuarentena (\d+)"
conciliacion_declarada = re.search(patron_gold, texto_informe)
COMPATIBLE_CON_GOLD = conciliacion_declarada is not None
assert COMPATIBLE_CON_GOLD, "el informe debe conservar la linea de conciliacion que Gold usa como compuerta previa"
bronze_declarado, silver_declarado, cuarentena_declarado = (
    int(conciliacion_declarada.group(1)), int(conciliacion_declarada.group(2)),
    int(conciliacion_declarada.group(3)))
COMPATIBLE_CON_GOLD = (bronze_declarado, silver_declarado, cuarentena_declarado) == (
    len(bronze_relectura), len(silver_file), len(quarantine_file))
assert COMPATIBLE_CON_GOLD, "los conteos declarados en el informe difieren de los CSV persistidos"

CONTROLES.append({
    "n": 21, "control": "informe_consistente_con_los_tres_csv", "resultado": True,
    "detalle": (f"el informe declara Bronze {bronze_declarado} = Silver {silver_declarado} + cuarentena "
                f"{cuarentena_declarado}, coincide con los tres CSV y conserva la linea de conciliacion "
                f"que Gold usa como compuerta previa")})
for nombre in ["silver", "quarantine", "reporte_calidad", "informe"]:
    CONTROLES.append({
        "n": 22, "control": f"informe_declara_el_entregable_{nombre}",
        "resultado": CONFIG["rutas"][nombre] in texto_informe,
        "detalle": CONFIG["rutas"][nombre]})
reglas_declaradas = re.search(r"Reglas evaluadas: (\d+)", texto_informe)
assert reglas_declaradas and int(reglas_declaradas.group(1)) == len(reporte_file), \
    "el numero de reglas del informe no coincide con el reporte de calidad"
CONTROLES.append({
    "n": 23, "control": "informe_declara_las_reglas_del_reporte", "resultado": True,
    "detalle": f"Reglas evaluadas: {reglas_declaradas.group(1)} igual a las filas del reporte de calidad"})
CONTROLES.append({
    "n": 24, "control": "informe_marca_la_copia_obsoleta_como_no_entregable",
    "resultado": ("No leido, no modificado y no entregable" in texto_informe
                  and CONFIG["rutas"]["copia_obsoleta_no_entregable"] in texto_informe),
    "detalle": "la copia obsoleta aparece solo como archivo excluido"})

tabla_controles = pd.DataFrame(CONTROLES)
print("Controles finales ampliados con la consistencia del informe")
print(tabla_controles.loc[tabla_controles["n"] >= 21, ["n", "control", "resultado", "detalle"]].to_string(index=False))
fallidos = tabla_controles.loc[~tabla_controles["resultado"], "control"].tolist()
assert not fallidos, f"controles fallidos: {fallidos}"
print()
print("CONTROLES_OK", len(tabla_controles), "controles en total, todos True")
print("REGLA_CONCILIACION_DECLARADA:", conciliacion_declarada.group(0))
print("Bronze no modificado al final:", sha256_de_archivo(PATHS["bronze"]) == BRONZE_SHA256_ANTES)
print("Copia obsoleta intacta:",
      (not COPIA_OBSOLETA_EXISTE)
      or sha256_de_archivo(PATHS["copia_obsoleta_no_entregable"]) == COPIA_OBSOLETA_SHA256_ANTES)

Controles finales ampliados con la consistencia del informe
 n                                                control  resultado                                                                                                                                                                                                                                                                                                                                                                        detalle
25 cierre_hallazgo_H1_banderas_informativas_en_cuarentena       True columna presente True; tokens conocidos True; numero de tokens igual al conteo en las 472 filas True; filas con conteo informativo mayor que cero y token vacio 0; filas con informativa 78; conjuntos por regla recalculados desde Bronze y las fuentes maestras coinciden True {'IOT-NUL-002': 2, 'IOT-REF-001': 76, 'IOT-REF-002': 50, 'IOT-REF-003': 22, 'IOT-SEQ-002': 0}
26                  cierre_hallazgo_H5_regla_no_aplicable       True